# TTAP aplicado a Talcahuano: HiGHS, Greedy Benefit y Greedy Density
Eduardo Carrasco Vidal — comparación sobre el mismo escenario GPS.

Se conservan las coordenadas, tareas, flota y función temporal del notebook
original. El objetivo inicial es **primary**, como en esa versión. El selector
permite multidimensional sólo si las tres cantidades representan ayuda
efectivamente entregada. Usar el mismo objetivo en A, B y Talcahuano.

Greedy Benefit maximiza el beneficio inmediato. Greedy Density maximiza ese
beneficio por minuto de vuelo hacia la tarea más servicio. Ambos reservan
tiempo para volver a base y cierran las rutas usadas, con el mismo desempate.
Las gráficas, métricas, rutas y balances comparan los tres métodos.

La cartografía GIS requiere los archivos locales originales. Está desactivada
inicialmente; las distancias GPS y los tres métodos funcionan sin esos archivos.
Activar `RUN_GIS_MAPS=True` para generar también los mapas originales.


In [ ]:
import numpy as np
import pandas as pd
import scipy
import matplotlib
import sys
import os
import platform

print("Computational Environment")
print("Python:", sys.version)
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)


from pathlib import Path
from IPython.display import display, SVG
import json


In [ ]:
# Objetivo:
# Comparar Greedy Benefit, Greedy Density y el mismo modelo exacto sobre la misma instancia TTAP.

import math
import time
from collections import deque, defaultdict
from typing import List
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import coo_matrix

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

### 1. Definición del problema:

In [ ]:
# El TTAP consiste en asignar helicópteros heterogéneos a tareas
# distribuidas espacialmente, maximizando el beneficio operacional
# acumulado bajo restricciones de:
#
# - compatibilidad helicóptero-tarea;
# - capacidades de carga, atención médica y traslado de personal;
# - tiempo de vuelo;
# - tiempo de servicio;
# - retorno a base y recuperación;
# - horizonte temporal;
# - ejecución única de cada tarea.
#
# En esta instancia estática:
# todas las tareas están disponibles desde t = 0.
#
# Beneficio humanitario temporal normalizado:
#
# b_i(t) = [w_i * eta_i * S_i(t)] / sum_j(w_j * eta_j)
#
# donde:
#
# eta_i = r_i / R_c(i) es el impacto humanitario normalizado;
# w_i es el peso de la clase de prioridad de la tarea;
# r_i es el impacto físico de la tarea y R_c(i) su referencia por clase;
#
# S_i(t) = 1, si t <= t_opt_i
# S_i(t) = 0, si t >= t_ineff_i
# Entre umbrales se usa la logística anclada implementada en satisfaction:
# S_i(t_opt)=1, S_i(t_eff)=0.5 y S_i(t_ineff)=0.
#
# ============================================================

SERVICE_TIME = 6.0
RECOVERY_TIME = 2.0
TMAX = 200.0
ALPHA = 0.1

USE_PRIORITY_WEIGHTS = True

PRIORITY_WEIGHTS = {
    "medical": 2.0,
    "personnel": 1.5,
    "cargo": 1.0,
}

# Referencias fijas de A y B para hacer eta_i adimensional y comparable.
IMPACT_REFERENCES = {
    "medical": 3.0,      # personas evacuadas médicamente
    "personnel": 3.0,    # personas transportadas
    "cargo": 500.0,      # kg entregados
}

EFFECTIVE_SATISFACTION_THRESHOLD = 0.5

# Discretización temporal del modelo exacto
TIME_UNIT = 1.0

# Criterio para no seleccionar tareas sin beneficio.
# Si una tarea tiene beneficio 0, no se considera útil para el greedy.
ZERO_BENEFIT_TOL = 0.0

# Escala común para gráficos de beneficio acumulado
BENEFIT_Y_MIN = 0.0
BENEFIT_Y_MAX = 0.70
BENEFIT_Y_STEP = 0.05


# Objetivo primary: sólo el resultado de la clase principal recibe beneficio.
# Multidimensional: pacientes, pasajeros no médicos y suministros son outputs distintos.
OBJECTIVE_MODE = 'primary'
if OBJECTIVE_MODE not in {'primary', 'multidimensional'}:
    raise ValueError('OBJECTIVE_MODE must be primary or multidimensional')
RUN_GIS_MAPS = False
EXACT_TIME_LIMIT = 180.0  # Segundos solicitados al solver; preparación se mide aparte.
EXACT_MIP_REL_GAP = 1e-6
PNG_DPI = 2000

plt.rcParams.update({'figure.dpi':160, 'font.family':'DejaVu Sans', 'font.size':8,
                     'axes.labelsize':8, 'axes.titlesize':9, 'xtick.labelsize':7,
                     'ytick.labelsize':7, 'pdf.fonttype':42, 'ps.fonttype':42,
                     'svg.fonttype':'none', 'lines.antialiased':True,
                     'lines.solid_capstyle':'round', 'lines.dash_capstyle':'round',
                     'axes.axisbelow':True})
METHOD_COLORS = {'Greedy Benefit':'#ff7f0e', 'Greedy Density':'#2ca02c'}

def save_publication_figure(fig, save_name, dpi=None):
    dpi = PNG_DPI if dpi is None else dpi
    if not np.isfinite(dpi) or dpi < 72:
        raise ValueError('PNG_DPI must be at least 72')
    width,height=fig.get_size_inches()
    if width*height*dpi*dpi*4 > 512*1024**2:
        raise ValueError('PNG canvas exceeds 512 MiB. Reduce its dimensions or PNG_DPI.')
    stem=Path(save_name).stem
    folder=Path(FIGURES_DIR);folder.mkdir(parents=True,exist_ok=True)
    for extension in ['pdf','svg','png']:
        fig.savefig(folder/(stem+'.'+extension), dpi=dpi, bbox_inches='tight', facecolor='white')
    preview=folder/'_preview';preview.mkdir(exist_ok=True)
    fig.savefig(preview/(stem+'.png'),dpi=240,bbox_inches='tight',facecolor='white')
    display(SVG(filename=str(folder/(stem+'.svg'))))
    plt.close(fig)

def save_current_figure_dual(save_name, dpi=None):
    save_publication_figure(plt.gcf(),save_name,dpi)


### 2. Datos del escenario:

In [ ]:
# ============================================================
# Configuración del escenario
# ============================================================

N_DEMAND_NODES = 7      # opciones recomendadas: 4 o 7
N_HELICOPTERS = 6       # opciones recomendadas: 4 o 6
N_TASKS = 30            # opciones: 20 o 30

SCENARIO_NAME = f"GPS_{N_DEMAND_NODES}areas_{N_HELICOPTERS}helos_{N_TASKS}tasks"

# Carpeta para guardar figuras del escenario
FIGURES_DIR = os.path.join("figures_ttap_GPS", SCENARIO_NAME + "_" + OBJECTIVE_MODE + "_three_methods")
os.makedirs(FIGURES_DIR, exist_ok=True)

# ============================================================
# Nodos GPS
# ============================================================

GPS_NODES_FULL = {
    "Base": (-36.69111351421891, -73.0478599920742),
    "A": (-36.70062541801277, -73.12141380985622), # Cancha Colegio Arturo Prat
    "B": (-36.738871104828384, -73.1102591903089), # Estacionamiento Hospital Higueras
    "C": (-36.75513285670206, -73.10765044444817), # Estadio Cap Huachipato
    "D": (-36.78339332263817, -73.09259088926588), # Club Hípico
    "E": (-36.784954466865976, -73.05085478019805), # Colegio Kingston College
    "F": (-36.83173804040988, -73.06322287960043), # Parque Bicentenario
    "G": (-36.81499794336835, -73.02359653839595), # Estadio Ester Roa
}

DEMAND_NODE_ORDER = ["A", "B", "C", "D", "E", "F", "G"]


def build_active_nodes(n_demand_nodes):
    if n_demand_nodes < 1 or n_demand_nodes > len(DEMAND_NODE_ORDER):
        raise ValueError(
            f"n_demand_nodes debe estar entre 1 y {len(DEMAND_NODE_ORDER)}"
        )

    active_names = ["Base"] + DEMAND_NODE_ORDER[:n_demand_nodes]

    active_nodes = {
        name: GPS_NODES_FULL[name]
        for name in active_names
    }

    return active_names, active_nodes


ACTIVE_NODE_NAMES, nodes = build_active_nodes(N_DEMAND_NODES)

# ============================================================
# Tareas base
# ============================================================

BASE_TASK_ROWS = [
    # task_id, visual_required, load, med, pers, t_opt, t_eff, t_ineff
    (1,  True,  200, 2, 1, 20, 40, 60),
    (2,  False, 300, 1, 2, 15, 30, 50),
    (3,  True,  100, 3, 0, 10, 25, 40),
    (4,  False, 200, 0, 3, 30, 50, 70),
    (5,  True,  150, 1, 1, 25, 45, 60),
    (6,  True,  100, 3, 0, 10, 25, 120),
    (7,  False, 400, 0, 3, 30, 50, 90),
    (8,  True,  150, 1, 1, 25, 45, 100),
    (9,  True,  200, 2, 1, 20, 40, 60),
    (10, False, 300, 1, 2, 15, 30, 50),
    (11, True,  100, 3, 0, 10, 25, 40),
    (12, False, 200, 0, 3, 30, 50, 70),
    (13, True,  150, 1, 1, 25, 45, 60),
    (14, True,  100, 3, 0, 10, 25, 120),
    (15, False, 400, 0, 3, 30, 50, 90),
    (16, True,  150, 1, 1, 25, 45, 100),
    (17, True,  150, 1, 1, 25, 45, 60),
    (18, True,  100, 3, 0, 10, 25, 120),
    (19, False, 400, 0, 3, 30, 50, 90),
    (20, True,  150, 1, 1, 25, 45, 100),
]

# Asignación espacial original de las 20 tareas base
BASE_TASK_TO_NODE = {}

for i in [1, 2, 9, 10]:
    BASE_TASK_TO_NODE[i] = "A"

for i in [3, 8, 7, 11, 12, 13, 14]:
    BASE_TASK_TO_NODE[i] = "B"

for i in [5, 6, 15, 16]:
    BASE_TASK_TO_NODE[i] = "C"

for i in [4, 17, 18, 19, 20]:
    BASE_TASK_TO_NODE[i] = "D"


def build_task_rows(n_tasks):
    """
    Construye el conjunto de tareas.
    - Si n_tasks = 20, usa exactamente las 20 tareas base.
    - Si n_tasks = 30, mantiene intactas las 20 tareas base
      y agrega 10 tareas nuevas replicando los perfiles de T1-T10.
    """

    if n_tasks not in [20, 30]:
        raise ValueError("Por ahora N_TASKS debe ser 20 o 30")

    rows = list(BASE_TASK_ROWS)

    if n_tasks == 30:
        additional_profiles = BASE_TASK_ROWS[:10]

        for new_id, base_row in zip(range(21, 31), additional_profiles):
            _, visual, load, med, pers, t_opt, t_eff, t_ineff = base_row

            rows.append(
                (
                    new_id,
                    visual,
                    load,
                    med,
                    pers,
                    t_opt,
                    t_eff,
                    t_ineff,
                )
            )

    return rows


def build_tasks(task_rows):
    return {
        row[0]: {
            "visual": row[1],
            "load": row[2],
            "med": row[3],
            "pers": row[4],
            "t_opt": row[5],
            "t_eff": row[6],
            "t_ineff": row[7],
        }
        for row in task_rows
    }


def build_task_to_node(tasks, active_node_names):
    """
    Mantiene intacta la asignación original de las tareas 1-20.
    Si existen tareas adicionales, se asignan preferentemente
    a los nodos nuevos E, F y G.
    """

    demand_nodes = [
        name for name in active_node_names
        if name != "Base"
    ]

    task_ids = sorted(tasks.keys())
    task_to_node = {}

    # ========================================================
    # 1. Mantener asignación original para tareas 1-20
    # ========================================================

    for task_id in task_ids:
        if task_id in BASE_TASK_TO_NODE:
            node_name = BASE_TASK_TO_NODE[task_id]

            if node_name not in demand_nodes:
                raise ValueError(
                    f"La tarea base {task_id} está asignada a {node_name}, "
                    f"pero ese nodo no está activo."
                )

            task_to_node[task_id] = node_name

    # ========================================================
    # 2. Asignar tareas adicionales, si existen
    # ========================================================

    additional_task_ids = [
        task_id for task_id in task_ids
        if task_id not in BASE_TASK_TO_NODE
    ]

    # Si existen E, F y G activos, las tareas nuevas se asignan ahí.
    new_task_nodes = [
        node_name for node_name in ["E", "F", "G"]
        if node_name in demand_nodes
    ]

    # Si no están activos E-G, se asignan a los nodos de demanda disponibles.
    if len(new_task_nodes) == 0:
        new_task_nodes = demand_nodes

    for idx, task_id in enumerate(additional_task_ids):
        node_name = new_task_nodes[idx % len(new_task_nodes)]
        task_to_node[task_id] = node_name

    # Control de consistencia
    missing_tasks = sorted(set(task_ids) - set(task_to_node.keys()))

    if len(missing_tasks) > 0:
        raise ValueError(
            f"Hay tareas sin nodo asignado: {missing_tasks}"
        )

    return task_to_node

task_rows = build_task_rows(N_TASKS)
tasks = build_tasks(task_rows)

def classify_task_priority(task):
    if task["med"] > 0:
        return "medical"

    if task["pers"] > 0:
        return "personnel"

    if task["load"] > 0:
        return "cargo"

    raise ValueError(
        "La tarea no tiene una necesidad operacional definida."
    )


def raw_humanitarian_impact(task, task_class):
    """Resultado físico usado para definir q_i; no es un costo de recursos."""
    if task_class == "medical":
        return float(task["med"]), "medical evacuees"

    if task_class == "personnel":
        return float(task["pers"]), "people transported"

    if task_class == "cargo":
        return float(task["load"]), "kg delivered"

    raise ValueError(f"Clase de tarea desconocida: {task_class}")


for task in tasks.values():
    priority_class = classify_task_priority(task)
    raw_impact, impact_unit = raw_humanitarian_impact(
        task,
        priority_class,
    )
    impact_reference = float(
        IMPACT_REFERENCES[priority_class]
    )

    if impact_reference <= 0:
        raise ValueError(
            "Todas las referencias de impacto deben ser positivas."
        )

    task["priority_class"] = priority_class
    task["class_weight"] = (
        float(PRIORITY_WEIGHTS[priority_class])
        if USE_PRIORITY_WEIGHTS
        else 1.0
    )

    # Alias de compatibilidad: mantiene intactas tablas y diagnósticos
    # posteriores que ya utilizan la columna weight.
    task["weight"] = task["class_weight"]

    task["raw_impact"] = raw_impact
    task["impact_unit"] = impact_unit
    task["impact_reference"] = impact_reference
    task["impact"] = raw_impact / impact_reference
    task["potential_value"] = (
        task["class_weight"] * task["impact"]
    )

    if OBJECTIVE_MODE == 'multidimensional':
        components = [PRIORITY_WEIGHTS['medical']*task['med']/IMPACT_REFERENCES['medical'],
                      PRIORITY_WEIGHTS['personnel']*task['pers']/IMPACT_REFERENCES['personnel'],
                      PRIORITY_WEIGHTS['cargo']*task['load']/IMPACT_REFERENCES['cargo']]
        if not USE_PRIORITY_WEIGHTS:
            components = [task['med']/IMPACT_REFERENCES['medical'],
                          task['pers']/IMPACT_REFERENCES['personnel'],
                          task['load']/IMPACT_REFERENCES['cargo']]
        task['potential_value'] = float(sum(components))
        task['class_weight'] = sum(PRIORITY_WEIGHTS.values()) if USE_PRIORITY_WEIGHTS else 3.0
        task['weight'] = task['class_weight']
        task['impact'] = task['potential_value']/task['class_weight']

    if not (0 < task["impact"] <= 1):
        raise ValueError(
            "El impacto normalizado debe cumplir 0 < q_i <= 1."
        )


TOTAL_POTENTIAL_VALUE = sum(
    task["potential_value"]
    for task in tasks.values()
)

if TOTAL_POTENTIAL_VALUE <= 0:
    raise ValueError(
        "El valor potencial total de la misión debe ser positivo."
    )


task_to_node = build_task_to_node(
    tasks,
    ACTIVE_NODE_NAMES
)

print(
    "Total potential mission value:",
    TOTAL_POTENTIAL_VALUE,
)

# ============================================================
# Helicópteros
# ============================================================

FLEET_ROWS = [
    ("H1", "H125_light",        True,  300, 1, 2, 190),
    ("H2", "H125_light",        False, 300, 1, 2, 190),

    ("H3", "AS365_Dauphin",     True,  500, 2, 4, 210),
    ("H4", "AS365_Dauphin",     False, 500, 2, 4, 210),

    ("H5", "H215M_Cougar",      True,  900, 3, 8, 200),
    ("H6", "H215M_Cougar",      False, 900, 3, 8, 200),
]


def build_helicopters(n_helicopters):
    if n_helicopters < 1 or n_helicopters > len(FLEET_ROWS):
        raise ValueError(
            f"n_helicopters debe estar entre 1 y {len(FLEET_ROWS)}"
        )

    selected_rows = FLEET_ROWS[:n_helicopters]

    return {
        h: {
            "model": model,
            "visual": visual,
            "load": load,
            "med": med,
            "pers": pers,
            "speed": speed,
        }
        for h, model, visual, load, med, pers, speed in selected_rows
    }


helicopters = build_helicopters(N_HELICOPTERS)

# ============================================================
# Control de configuración
# ============================================================

print("=" * 70)
print("SCENARIO CONFIGURATION")
print("=" * 70)

print(f"Scenario name     : {SCENARIO_NAME}")
print(f"Demand nodes      : {N_DEMAND_NODES}")
print(f"Active nodes      : {ACTIVE_NODE_NAMES}")
print(f"Helicopters       : {N_HELICOPTERS}")
print(f"Tasks             : {N_TASKS}")
print(f"Figures directory : {FIGURES_DIR}")

print("\nTasks per node:")
task_distribution = (
    pd.Series(task_to_node)
    .value_counts()
    .sort_index()
)

display(task_distribution.to_frame("Number of tasks"))

print("\nTask-to-node assignment:")
display(
    pd.DataFrame({
        "Task ID": list(task_to_node.keys()),
        "Node": list(task_to_node.values())
    }).sort_values("Task ID")
)

print("\nFleet:")
display(pd.DataFrame(helicopters).T)

### 3. Funciones comunes:

In [ ]:
def haversine_distance_km(node_a, node_b):
    """
    Calcula la distancia geodésica aproximada entre dos nodos GPS
    usando la fórmula de Haversine.

    Las coordenadas deben estar en formato:
    (latitud, longitud)

    Retorna distancia en kilómetros.
    """

    lat1, lon1 = nodes[node_a]
    lat2, lon2 = nodes[node_b]

    radius_earth_km = 6371.0088

    lat1_rad = math.radians(lat1)
    lon1_rad = math.radians(lon1)
    lat2_rad = math.radians(lat2)
    lon2_rad = math.radians(lon2)

    delta_lat = lat2_rad - lat1_rad
    delta_lon = lon2_rad - lon1_rad

    a = (
        math.sin(delta_lat / 2.0) ** 2
        + math.cos(lat1_rad)
        * math.cos(lat2_rad)
        * math.sin(delta_lon / 2.0) ** 2
    )

    c = 2.0 * math.asin(math.sqrt(a))

    return radius_earth_km * c
    

def travel_time_minutes(h, node_a, node_b):
    if node_a == node_b:
        return 0.0

    distance_km = haversine_distance_km(node_a, node_b)
    speed_kmh = helicopters[h]["speed"]

    return 60.0 * distance_km / speed_kmh


def logistic_core(t, t_eff):
    return 1.0 / (
        1.0 + math.exp(ALPHA * (t - t_eff))
    )


def satisfaction(t, task):
    t_opt = task["t_opt"]
    t_eff = task["t_eff"]
    t_ineff = task["t_ineff"]

    if not (t_opt < t_eff < t_ineff):
        raise ValueError(
            "Los umbrales deben cumplir "
            "t_opt < t_eff < t_ineff."
        )

    if t <= t_opt:
        return 1.0

    if t >= t_ineff:
        return 0.0

    logistic_t = logistic_core(t, t_eff)
    logistic_eff = 0.5

    if t <= t_eff:
        logistic_opt = logistic_core(
            t_opt,
            t_eff
        )

        value = (
            0.5
            + 0.5
            * (logistic_t - logistic_eff)
            / (logistic_opt - logistic_eff)
        )
    else:
        logistic_ineff = logistic_core(
            t_ineff,
            t_eff
        )

        value = (
            0.5
            * (logistic_t - logistic_ineff)
            / (logistic_eff - logistic_ineff)
        )

    return max(0.0, min(1.0, value))


def task_benefit(task_id, finish_time):
    task = tasks[task_id]

    return (
        task["potential_value"]
        * satisfaction(finish_time, task)
        / TOTAL_POTENTIAL_VALUE
    )


# Invariante de normalización: si todas las tareas terminan en t=0,
# el valor humanitario acumulado de la misión es exactamente 1.
assert np.isclose(
    sum(
        task_benefit(task_id, 0.0)
        for task_id in tasks
    ),
    1.0,
    atol=1e-12,
)

def ceil_steps(minutes, time_unit):
    return int(math.ceil((minutes - 1e-9) / time_unit))

def time_from_step(k, time_unit):
    return k * time_unit

def can_execute_task_by_resources(h, task_id, load, med, pers):
    helo = helicopters[h]
    task = tasks[task_id]

    if task["visual"] and not helo["visual"]:
        return False

    if load < task["load"]:
        return False

    if med < task["med"]:
        return False

    if pers < task["pers"]:
        return False

    return True


def helo_to_str(h):
    if isinstance(h, str):
        h = h.strip()
        if h.startswith("H"):
            return h
        return f"H{h}"
    return f"H{int(h)}"


def helo_to_int(h):
    if isinstance(h, str):
        h = h.strip()
        if h.startswith("H"):
            return int(h.replace("H", ""))
        return int(h)
    return int(h)

## Prueba para distancias GPS:

In [ ]:
print("GPS nodes activos:")
for node_name, coord in nodes.items():
    print(node_name, coord)

print("\nDistancias desde Base:")
for node_name in nodes:
    if node_name == "Base":
        continue

    d_km = haversine_distance_km("Base", node_name)

    print(
        f"Base → {node_name}: "
        f"{d_km:.3f} km"
    )

In [ ]:
if RUN_GIS_MAPS:
    import geopandas as gpd
    
    COMUNAS_PATH = "data/limites_comunales/comunas_final.zip"
    
    comunas = gpd.read_file(COMUNAS_PATH)
    
    print(comunas.shape)
    print(comunas.crs)
    print(comunas.columns)
    display(comunas.head())

else:
    print('Mapas GIS omitidos; activar RUN_GIS_MAPS con los archivos locales originales.')


In [ ]:
if RUN_GIS_MAPS:
    import os
    import zipfile
    from pathlib import Path
    
    import geopandas as gpd
    import pandas as pd
    import matplotlib.pyplot as plt
    import contextily as ctx
    
    # ============================================================
    # Lectura de carta de inundación por tsunami desde KMZ
    # ============================================================
    
    TSUNAMI_KMZ_PATH = "data/tsunami/CITSU_Bahias_Concepcion_San Vicente_1ra_Ed2013.kmz"
    
    
    def read_kmz_as_geodataframe(kmz_path):
        """
        Lee un archivo KMZ que contiene un KML interno y lo devuelve
        como GeoDataFrame.
        """
    
        kmz_path = Path(kmz_path)
    
        if not kmz_path.exists():
            raise FileNotFoundError(
                f"No existe el archivo KMZ en:\n{kmz_path}"
            )
    
        extract_dir = kmz_path.parent / f"{kmz_path.stem}_extracted"
        extract_dir.mkdir(parents=True, exist_ok=True)
    
        with zipfile.ZipFile(kmz_path, "r") as z:
            z.extractall(extract_dir)
    
        kml_files = list(extract_dir.rglob("*.kml"))
    
        if len(kml_files) == 0:
            raise FileNotFoundError(
                f"No se encontró ningún archivo KML dentro de:\n{kmz_path}"
            )
    
        kml_path = kml_files[0]
    
        gdf = gpd.read_file(kml_path)
    
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4326)
    
        return gdf
    
    
    tsunami_gdf = read_kmz_as_geodataframe(TSUNAMI_KMZ_PATH)
    
    # Limpiar geometrías vacías
    tsunami_gdf = tsunami_gdf[
        tsunami_gdf.geometry.notna() & ~tsunami_gdf.geometry.is_empty
    ].copy()
    
    print("Carta de inundación cargada")
    print("Shape:", tsunami_gdf.shape)
    print("CRS:", tsunami_gdf.crs)
    display(tsunami_gdf[["Name", "geometry"]].head())


In [ ]:
if RUN_GIS_MAPS:
    # ============================================================
    # Crear área total de inundación
    # ============================================================
    
    # La carta trae varias geometrías por profundidad.
    # Para el mapa principal, las unimos en una sola capa roja.
    tsunami_area = tsunami_gdf.dissolve()
    
    # Asegurar CRS geográfico de origen
    if tsunami_area.crs is None:
        tsunami_area = tsunami_area.set_crs(epsg=4326)
    
    # Reproyectar a Web Mercator para usar con contextily
    tsunami_area_3857 = tsunami_area.to_crs(epsg=3857)


In [ ]:
if RUN_GIS_MAPS:
    import os
    import unicodedata
    
    import geopandas as gpd
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patheffects as path_effects
    from shapely.geometry import Point
    import contextily as ctx
    
    # ============================================================
    # Parámetros del mapa
    # ============================================================
    
    MAP_NAME_TSUNAMI = "ttap_gps_area_map_comunas_tsunami_chart"
    MAP_ZOOM_BASE = 12
    MAP_ZOOM_LABELS = 11
    FIG_SIZE = (8, 8)
    PADDING_FACTOR = 0.12
    
    # Ruta al shapefile comprimido
    COMUNAS_PATH = "data/limites_comunales/comunas_final.zip"
    
    if "FIGURES_DIR" not in globals():
        FIGURES_DIR = "figures_ttap_GPS"
    
    os.makedirs(FIGURES_DIR, exist_ok=True)
    
    # Comunas a mostrar
    AREA_COMUNAS = [
        "Talcahuano",
        "Concepción",
        "Hualpén",
        #"Penco",
        #"Tomé",
        #"San Pedro de la Paz",
        #"Chiguayante",
    ]
    
    # ============================================================
    # Datos GPS de nodos TTAP
    # ============================================================
    
    GPS_NODES_FULL = {
        "Base": (-36.69111351421891, -73.0478599920742),
        "A": (-36.70062541801277, -73.12141380985622), # Cancha Colegio Arturo Prat
        "B": (-36.738871104828384, -73.1102591903089), # Estacionamiento Hospital Higueras
        "C": (-36.75513285670206, -73.10765044444817), # Estadio Cap Huachipato
        "D": (-36.78339332263817, -73.09259088926588), # Club Hípico
        "E": (-36.784954466865976, -73.05085478019805), # Colegio Kingston College
        "F": (-36.83173804040988, -73.06322287960043), # Parque Bicentenario
        "G": (-36.81499794336835, -73.02359653839595), # Estadio Ester Roa
    }
    
    ACTIVE_NODE_NAMES = ["Base", "A", "B", "C", "D", "E", "F", "G"]
    
    # ============================================================
    # Funciones auxiliares
    # ============================================================
    
    def normalize_text(value):
        """
        Normaliza texto para comparar nombres con o sin tilde.
        """
    
        if pd.isna(value):
            return ""
    
        value = str(value).strip().lower()
    
        value = "".join(
            c for c in unicodedata.normalize("NFD", value)
            if unicodedata.category(c) != "Mn"
        )
    
        return value
    
    
    def read_comunas(path):
        """
        Lee el archivo de comunas.
        Intenta leer directamente el .zip y, si falla, usa zip://.
        """
    
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"No existe el archivo de comunas en:\n{path}"
            )
    
        try:
            return gpd.read_file(path)
        except Exception:
            return gpd.read_file(f"zip://{path}")
    
    
    def find_comuna_column(gdf):
        """
        Detecta la columna que contiene el nombre de la comuna.
        En tu archivo debería ser 'Comuna'.
        """
    
        candidate_columns = [
            "Comuna",
            "COMUNA",
            "comuna",
            "NOM_COMUNA",
            "nom_comuna",
            "NOMBRE_COM",
            "nombre_com",
            "NOMBRE",
            "nombre",
        ]
    
        for col in candidate_columns:
            if col in gdf.columns:
                return col
    
        raise ValueError(
            "No pude detectar la columna de nombre de comuna. "
            f"Columnas disponibles: {list(gdf.columns)}"
        )
    
    
    # ============================================================
    # Cargar límites comunales
    # ============================================================
    
    comunas = read_comunas(COMUNAS_PATH)
    
    if comunas.crs is None:
        raise ValueError(
            "El archivo de comunas no tiene CRS definido. "
            "Debes asignar el CRS correcto antes de graficar."
        )
    
    comuna_col = find_comuna_column(comunas)
    
    comunas["_comuna_norm"] = comunas[comuna_col].apply(normalize_text)
    
    area_comunas_norm = [
        normalize_text(name)
        for name in AREA_COMUNAS
    ]
    
    comunas_area = comunas[
        comunas["_comuna_norm"].isin(area_comunas_norm)
    ].copy()
    
    if comunas_area.empty:
        raise ValueError(
            "No se encontraron las comunas indicadas. "
            "Revisa AREA_COMUNAS y la columna de nombres."
        )
    
    print("CRS comunas:", comunas.crs)
    print("Columna usada para comuna:", comuna_col)
    print("Comunas encontradas:")
    display(comunas_area[[comuna_col]])
    
    # ============================================================
    # Crear GeoDataFrame de nodos TTAP
    # ============================================================
    
    node_rows = []
    
    for name in ACTIVE_NODE_NAMES:
        lat, lon = GPS_NODES_FULL[name]
    
        node_rows.append({
            "name": name,
            "lat": lat,
            "lon": lon,
            "type": "Base" if name == "Base" else "Demand node",
            "geometry": Point(lon, lat),
        })
    
    gdf_nodes = gpd.GeoDataFrame(
        node_rows,
        crs="EPSG:4326"
    )
    
    # ============================================================
    # Reproyección a Web Mercator
    # ============================================================
    # Contextily usa mapas base en EPSG:3857.
    # Por eso dejamos comunas y nodos en el mismo CRS.
    
    comunas_3857 = comunas_area.to_crs(epsg=3857)
    nodes_3857 = gdf_nodes.to_crs(epsg=3857)
    
    # ============================================================
    # Figura
    # ============================================================
    
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    
    # ============================================================
    # Extensión del mapa
    # ============================================================
    
    # límites basados en los nodos, no en toda la geometría comunal
    xmin, ymin, xmax, ymax = nodes_3857.total_bounds
    
    # padding en metros (ajústalo a gusto)
    padding_x = 3500
    padding_y = 3500
    
    ax.set_xlim(-8.151e6, -8.116e6)
    ax.set_ylim(-4.420e6, -4.390e6)
    
    # ============================================================
    # Límites comunales, estilo GeoPandas gallery
    # ============================================================
    # Esto es lo que genera el efecto de "áreas azules delineadas".
    
    comunas_3857.plot(
        ax=ax,
        color="tab:blue",
        alpha=0.32,
        edgecolor="black",
        linewidth=0.8,
        zorder=2,
        label="Communal boundaries"
    )
    
    # ============================================================
    # Mapa base
    # ============================================================
    # Primero agregamos el fondo sin etiquetas.
    # Luego agregamos solo las etiquetas encima, como sugiere la galería.
    
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.PositronNoLabels,
        zoom=MAP_ZOOM_BASE,
        reset_extent=False,
        attribution_size=5
    )
    
    # Volvemos a dibujar los bordes comunales encima del basemap,
    # para asegurar que queden visibles.
    comunas_3857.boundary.plot(
        ax=ax,
        color="black",
        linewidth=0.8,
        zorder=3
    )
    
    # Relleno comunal translúcido encima.
    comunas_3857.plot(
        ax=ax,
        color="tab:blue",
        alpha=0.25,
        edgecolor="black",
        linewidth=0.8,
        zorder=4,
        label="Communal boundaries"
    )
    
    # ============================================================
    # Carta de inundación por tsunami
    # ============================================================
    
    tsunami_area_3857.plot(
        ax=ax,
        color="red",
        alpha=0.28,
        edgecolor="red",
        linewidth=1.0,
        zorder=5,
        label="Tsunami inundation area"
    )
    
    # Borde más marcado de la zona de inundación
    tsunami_area_3857.boundary.plot(
        ax=ax,
        color="red",
        linewidth=1.1,
        alpha=0.30,
        zorder=6
    )
    
    # ============================================================
    # Etiquetas de comunas
    # ============================================================
    
    for _, row in comunas_3857.iterrows():
        point = row.geometry.representative_point()
        label = row[comuna_col]
    
        text = ax.annotate(
            label,
            xy=(point.x, point.y),
            ha="center",
            va="center",
            fontsize=7,
            color="dimgray",
            zorder=9
        )
    
        text.set_path_effects([
            path_effects.Stroke(linewidth=2.5, foreground="white"),
            path_effects.Normal()
        ])
    
    # ============================================================
    # Nodos TTAP
    # ============================================================
    
    nodes_3857[nodes_3857["type"] == "Demand node"].plot(
        ax=ax,
        markersize=90,
        marker="o",
        color="white",
        edgecolor="black",
        linewidth=1.0,
        alpha=0.90,
        zorder=8,
        label="Demand nodes"
    )
    
    nodes_3857[nodes_3857["type"] == "Base"].plot(
        ax=ax,
        markersize=210,
        marker="*",
        color="darkred",
        edgecolor="black",
        linewidth=0.9,
        alpha=0.98,
        zorder=8,
        label="Base"
    )
    
    # ============================================================
    # Etiquetas de nodos
    # ============================================================
    
    label_offsets = {
        "Base": (9, 9),
        "A": (8, 6),
        "B": (8, 6),
        "C": (8, 6),
        "D": (8, 6),
        "E": (8, 6),
        "F": (8, 6),
        "G": (8, 6),
    }
    
    for _, row in nodes_3857.iterrows():
    
        dx, dy = label_offsets.get(row["name"], (8, 6))
    
        if row["name"] == "Base":
            label_color = "darkred"
            font_size = 12
        else:
            label_color = "black"
            font_size = 11
    
        text = ax.annotate(
            row["name"],
            xy=(row.geometry.x, row.geometry.y),
            xytext=(dx, dy),
            textcoords="offset points",
            color=label_color,
            fontweight="bold",
            fontsize=font_size,
            zorder=8
        )
    
        text.set_path_effects([
            path_effects.Stroke(linewidth=2.8, foreground="white"),
            path_effects.Normal()
        ])
    
    # ============================================================
    # Estética final
    # ============================================================
    
    ax.set_title(
        "TTAP demand nodes and tsunami inundation area",
        fontsize=12,
        fontweight="bold",
        pad=12
    )
    
    # Quitar valores de los ejes
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Quitar marcas de los ejes
    ax.tick_params(
        left=False,
        bottom=False,
        labelleft=False,
        labelbottom=False
    )
    
    plt.tight_layout()
    
    # ============================================================
    # Guardado
    # ============================================================
    
    png_path = os.path.join(FIGURES_DIR, f"{MAP_NAME_TSUNAMI}.png")
    pdf_path = os.path.join(FIGURES_DIR, f"{MAP_NAME_TSUNAMI}.pdf")
    
    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )
    
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )
    
    print(f"Map saved at: {png_path}")
    print(f"Map saved at: {pdf_path}")
    
    plt.show()


In [ ]:
if RUN_GIS_MAPS:
    import os
    import unicodedata
    
    import geopandas as gpd
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patheffects as path_effects
    from shapely.geometry import Point
    import contextily as ctx
    
    # ============================================================
    # Parámetros del mapa
    # ============================================================
    
    MAP_NAME = "ttap_gps_area_map_comunas"
    MAP_ZOOM_BASE = 12
    MAP_ZOOM_LABELS = 11
    FIG_SIZE = (8, 8)
    PADDING_FACTOR = 0.12
    
    # Ruta al shapefile comprimido
    COMUNAS_PATH = "data/limites_comunales/comunas_final.zip"
    
    if "FIGURES_DIR" not in globals():
        FIGURES_DIR = "figures_ttap_GPS"
    
    os.makedirs(FIGURES_DIR, exist_ok=True)
    
    # Comunas a mostrar
    AREA_COMUNAS = [
        "Talcahuano",
        "Concepción",
        "Hualpén",
        #"Penco",
        #"Tomé",
        #"San Pedro de la Paz",
        #"Chiguayante",
    ]
    
    # ============================================================
    # Datos GPS de nodos TTAP
    # ============================================================
    
    GPS_NODES_FULL = {
        "Base": (-36.69111351421891, -73.0478599920742),
        "A": (-36.70062541801277, -73.12141380985622), # Cancha Colegio Arturo Prat
        "B": (-36.738871104828384, -73.1102591903089), # Estacionamiento Hospital Higueras
        "C": (-36.75513285670206, -73.10765044444817), # Estadio Cap Huachipato
        "D": (-36.78339332263817, -73.09259088926588), # Club Hípico
        "E": (-36.784954466865976, -73.05085478019805), # Colegio Kingston College
        "F": (-36.83173804040988, -73.06322287960043), # Parque Bicentenario
        "G": (-36.81499794336835, -73.02359653839595), # Estadio Ester Roa
    }
    
    ACTIVE_NODE_NAMES = ["Base", "A", "B", "C", "D", "E", "F", "G"]
    
    # ============================================================
    # Funciones auxiliares
    # ============================================================
    
    def normalize_text(value):
        """
        Normaliza texto para comparar nombres con o sin tilde.
        """
    
        if pd.isna(value):
            return ""
    
        value = str(value).strip().lower()
    
        value = "".join(
            c for c in unicodedata.normalize("NFD", value)
            if unicodedata.category(c) != "Mn"
        )
    
        return value
    
    
    def read_comunas(path):
        """
        Lee el archivo de comunas.
        Intenta leer directamente el .zip y, si falla, usa zip://.
        """
    
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"No existe el archivo de comunas en:\n{path}"
            )
    
        try:
            return gpd.read_file(path)
        except Exception:
            return gpd.read_file(f"zip://{path}")
    
    
    def find_comuna_column(gdf):
        """
        Detecta la columna que contiene el nombre de la comuna.
        En tu archivo debería ser 'Comuna'.
        """
    
        candidate_columns = [
            "Comuna",
            "COMUNA",
            "comuna",
            "NOM_COMUNA",
            "nom_comuna",
            "NOMBRE_COM",
            "nombre_com",
            "NOMBRE",
            "nombre",
        ]
    
        for col in candidate_columns:
            if col in gdf.columns:
                return col
    
        raise ValueError(
            "No pude detectar la columna de nombre de comuna. "
            f"Columnas disponibles: {list(gdf.columns)}"
        )
    
    
    # ============================================================
    # Cargar límites comunales
    # ============================================================
    
    comunas = read_comunas(COMUNAS_PATH)
    
    if comunas.crs is None:
        raise ValueError(
            "El archivo de comunas no tiene CRS definido. "
            "Debes asignar el CRS correcto antes de graficar."
        )
    
    comuna_col = find_comuna_column(comunas)
    
    comunas["_comuna_norm"] = comunas[comuna_col].apply(normalize_text)
    
    area_comunas_norm = [
        normalize_text(name)
        for name in AREA_COMUNAS
    ]
    
    comunas_area = comunas[
        comunas["_comuna_norm"].isin(area_comunas_norm)
    ].copy()
    
    if comunas_area.empty:
        raise ValueError(
            "No se encontraron las comunas indicadas. "
            "Revisa AREA_COMUNAS y la columna de nombres."
        )
    
    print("CRS comunas:", comunas.crs)
    print("Columna usada para comuna:", comuna_col)
    print("Comunas encontradas:")
    display(comunas_area[[comuna_col]])
    
    # ============================================================
    # Crear GeoDataFrame de nodos TTAP
    # ============================================================
    
    node_rows = []
    
    for name in ACTIVE_NODE_NAMES:
        lat, lon = GPS_NODES_FULL[name]
    
        node_rows.append({
            "name": name,
            "lat": lat,
            "lon": lon,
            "type": "Base" if name == "Base" else "Demand node",
            "geometry": Point(lon, lat),
        })
    
    gdf_nodes = gpd.GeoDataFrame(
        node_rows,
        crs="EPSG:4326"
    )
    
    # ============================================================
    # Reproyección a Web Mercator
    # ============================================================
    # Contextily usa mapas base en EPSG:3857.
    # Por eso dejamos comunas y nodos en el mismo CRS.
    
    comunas_3857 = comunas_area.to_crs(epsg=3857)
    nodes_3857 = gdf_nodes.to_crs(epsg=3857)
    
    # ============================================================
    # Figura
    # ============================================================
    
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    
    # ============================================================
    # Extensión del mapa
    # ============================================================
    
    # límites basados en los nodos, no en toda la geometría comunal
    xmin, ymin, xmax, ymax = nodes_3857.total_bounds
    
    # padding en metros (ajústalo a gusto)
    padding_x = 3500
    padding_y = 3500
    
    ax.set_xlim(-8.151e6, -8.116e6)
    ax.set_ylim(-4.420e6, -4.390e6)
    
    # ============================================================
    # Límites comunales, estilo GeoPandas gallery
    # ============================================================
    # Esto es lo que genera el efecto de "áreas azules delineadas".
    
    comunas_3857.plot(
        ax=ax,
        color="tab:blue",
        alpha=0.32,
        edgecolor="black",
        linewidth=0.8,
        zorder=2,
        label="Communal boundaries"
    )
    
    # ============================================================
    # Mapa base
    # ============================================================
    # Primero agregamos el fondo sin etiquetas.
    # Luego agregamos solo las etiquetas encima, como sugiere la galería.
    
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.PositronNoLabels,
        zoom=MAP_ZOOM_BASE,
        reset_extent=False,
        attribution_size=5
    )
    
    # Volvemos a dibujar los bordes comunales encima del basemap,
    # para asegurar que queden visibles.
    comunas_3857.boundary.plot(
        ax=ax,
        color="black",
        linewidth=0.8,
        zorder=3
    )
    
    # Relleno comunal translúcido encima.
    comunas_3857.plot(
        ax=ax,
        color="tab:blue",
        alpha=0.25,
        edgecolor="black",
        linewidth=0.8,
        zorder=4,
        label="Communal boundaries"
    )
    
    # ============================================================
    # Etiquetas de comunas
    # ============================================================
    
    for _, row in comunas_3857.iterrows():
        point = row.geometry.representative_point()
        label = row[comuna_col]
    
        text = ax.annotate(
            label,
            xy=(point.x, point.y),
            ha="center",
            va="center",
            fontsize=7,
            color="dimgray",
            zorder=9
        )
    
        text.set_path_effects([
            path_effects.Stroke(linewidth=2.5, foreground="white"),
            path_effects.Normal()
        ])
    
    # ============================================================
    # Nodos TTAP
    # ============================================================
    
    nodes_3857[nodes_3857["type"] == "Demand node"].plot(
        ax=ax,
        markersize=90,
        marker="o",
        color="white",
        edgecolor="black",
        linewidth=1.0,
        alpha=0.90,
        zorder=8,
        label="Demand nodes"
    )
    
    nodes_3857[nodes_3857["type"] == "Base"].plot(
        ax=ax,
        markersize=210,
        marker="*",
        color="darkred",
        edgecolor="black",
        linewidth=0.9,
        alpha=0.98,
        zorder=8,
        label="Base"
    )
    
    # ============================================================
    # Etiquetas de nodos
    # ============================================================
    
    label_offsets = {
        "Base": (9, 9),
        "A": (8, 6),
        "B": (8, 6),
        "C": (8, 6),
        "D": (8, 6),
        "E": (8, 6),
        "F": (8, 6),
        "G": (8, 6),
    }
    
    for _, row in nodes_3857.iterrows():
    
        dx, dy = label_offsets.get(row["name"], (8, 6))
    
        if row["name"] == "Base":
            label_color = "darkred"
            font_size = 12
        else:
            label_color = "black"
            font_size = 11
    
        text = ax.annotate(
            row["name"],
            xy=(row.geometry.x, row.geometry.y),
            xytext=(dx, dy),
            textcoords="offset points",
            color=label_color,
            fontweight="bold",
            fontsize=font_size,
            zorder=8
        )
    
        text.set_path_effects([
            path_effects.Stroke(linewidth=2.8, foreground="white"),
            path_effects.Normal()
        ])
    
    # ============================================================
    # Estética final
    # ============================================================
    
    ax.set_title(
        "TTAP demand nodes and tsunami inundation area",
        fontsize=12,
        fontweight="bold",
        pad=12
    )
    
    # Quitar valores de los ejes
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Quitar marcas de los ejes
    ax.tick_params(
        left=False,
        bottom=False,
        labelleft=False,
        labelbottom=False
    )
    
    plt.tight_layout()
    
    # ============================================================
    # Guardado
    # ============================================================
    
    png_path = os.path.join(FIGURES_DIR, f"{MAP_NAME}.png")
    pdf_path = os.path.join(FIGURES_DIR, f"{MAP_NAME}.pdf")
    
    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )
    
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )
    
    print(f"Map saved at: {png_path}")
    print(f"Map saved at: {pdf_path}")
    
    plt.show()


In [ ]:
if RUN_GIS_MAPS:
    EVAC_POINTS_KMZ_PATH = "data/puntos_evacuacion/puntos_de_evacuacion.kmz"
    
    evac_points_gdf = read_kmz_as_geodataframe(EVAC_POINTS_KMZ_PATH)
    
    evac_points_gdf = evac_points_gdf[
        evac_points_gdf.geometry.notna() & ~evac_points_gdf.geometry.is_empty
    ].copy()
    
    print("Puntos de evacuación cargados")
    print("Shape:", evac_points_gdf.shape)
    print("CRS:", evac_points_gdf.crs)
    display(evac_points_gdf[["Name", "geometry"]].head())


In [ ]:
if RUN_GIS_MAPS:
    from shapely.geometry import Point
    
    evac_points_gdf["geometry"] = evac_points_gdf["geometry"].apply(
        lambda geom: Point(geom.x, geom.y) if geom is not None else None
    )


In [ ]:
if RUN_GIS_MAPS:
    import os
    import unicodedata
    
    import geopandas as gpd
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.patheffects as path_effects
    from shapely.geometry import Point
    import contextily as ctx
    
    # ============================================================
    # Parámetros del mapa
    # ============================================================
    
    MAP_NAME_TSUNAMI = "ttap_gps_area_map_comunas_tsunami_chart_ep"
    MAP_ZOOM_BASE = 12
    MAP_ZOOM_LABELS = 11
    FIG_SIZE = (8, 8)
    PADDING_FACTOR = 0.12
    
    # Ruta al shapefile comprimido
    COMUNAS_PATH = "data/limites_comunales/comunas_final.zip"
    
    if "FIGURES_DIR" not in globals():
        FIGURES_DIR = "figures_ttap_GPS"
    
    os.makedirs(FIGURES_DIR, exist_ok=True)
    
    # Comunas a mostrar
    AREA_COMUNAS = [
        "Talcahuano",
        "Concepción",
        "Hualpén",
        #"Penco",
        #"Tomé",
        #"San Pedro de la Paz",
        #"Chiguayante",
    ]
    
    # ============================================================
    # Datos GPS de nodos TTAP
    # ============================================================
    
    GPS_NODES_FULL = {
        "Base": (-36.69111351421891, -73.0478599920742),
        "A": (-36.70062541801277, -73.12141380985622), # Cancha Colegio Arturo Prat
        "B": (-36.738871104828384, -73.1102591903089), # Estacionamiento Hospital Higueras
        "C": (-36.75513285670206, -73.10765044444817), # Estadio Cap Huachipato
        "D": (-36.78339332263817, -73.09259088926588), # Club Hípico
        "E": (-36.784954466865976, -73.05085478019805), # Colegio Kingston College
        "F": (-36.83173804040988, -73.06322287960043), # Parque Bicentenario
        "G": (-36.81499794336835, -73.02359653839595), # Estadio Ester Roa
    }
    
    ACTIVE_NODE_NAMES = ["Base", "A", "B", "C", "D", "E", "F", "G"]
    
    # ============================================================
    # Funciones auxiliares
    # ============================================================
    
    def normalize_text(value):
        """
        Normaliza texto para comparar nombres con o sin tilde.
        """
    
        if pd.isna(value):
            return ""
    
        value = str(value).strip().lower()
    
        value = "".join(
            c for c in unicodedata.normalize("NFD", value)
            if unicodedata.category(c) != "Mn"
        )
    
        return value
    
    
    def read_comunas(path):
        """
        Lee el archivo de comunas.
        Intenta leer directamente el .zip y, si falla, usa zip://.
        """
    
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"No existe el archivo de comunas en:\n{path}"
            )
    
        try:
            return gpd.read_file(path)
        except Exception:
            return gpd.read_file(f"zip://{path}")
    
    
    def find_comuna_column(gdf):
        """
        Detecta la columna que contiene el nombre de la comuna.
        En tu archivo debería ser 'Comuna'.
        """
    
        candidate_columns = [
            "Comuna",
            "COMUNA",
            "comuna",
            "NOM_COMUNA",
            "nom_comuna",
            "NOMBRE_COM",
            "nombre_com",
            "NOMBRE",
            "nombre",
        ]
    
        for col in candidate_columns:
            if col in gdf.columns:
                return col
    
        raise ValueError(
            "No pude detectar la columna de nombre de comuna. "
            f"Columnas disponibles: {list(gdf.columns)}"
        )
    
    
    # ============================================================
    # Cargar límites comunales
    # ============================================================
    
    comunas = read_comunas(COMUNAS_PATH)
    
    if comunas.crs is None:
        raise ValueError(
            "El archivo de comunas no tiene CRS definido. "
            "Debes asignar el CRS correcto antes de graficar."
        )
    
    comuna_col = find_comuna_column(comunas)
    
    comunas["_comuna_norm"] = comunas[comuna_col].apply(normalize_text)
    
    area_comunas_norm = [
        normalize_text(name)
        for name in AREA_COMUNAS
    ]
    
    comunas_area = comunas[
        comunas["_comuna_norm"].isin(area_comunas_norm)
    ].copy()
    
    if comunas_area.empty:
        raise ValueError(
            "No se encontraron las comunas indicadas. "
            "Revisa AREA_COMUNAS y la columna de nombres."
        )
    
    print("CRS comunas:", comunas.crs)
    print("Columna usada para comuna:", comuna_col)
    print("Comunas encontradas:")
    display(comunas_area[[comuna_col]])
    
    # ============================================================
    # Crear GeoDataFrame de nodos TTAP
    # ============================================================
    
    node_rows = []
    
    for name in ACTIVE_NODE_NAMES:
        lat, lon = GPS_NODES_FULL[name]
    
        node_rows.append({
            "name": name,
            "lat": lat,
            "lon": lon,
            "type": "Base" if name == "Base" else "Demand node",
            "geometry": Point(lon, lat),
        })
    
    gdf_nodes = gpd.GeoDataFrame(
        node_rows,
        crs="EPSG:4326"
    )
    
    # ============================================================
    # Reproyección a Web Mercator
    # ============================================================
    # Contextily usa mapas base en EPSG:3857.
    # Por eso dejamos comunas y nodos en el mismo CRS.
    
    comunas_3857 = comunas_area.to_crs(epsg=3857)
    nodes_3857 = gdf_nodes.to_crs(epsg=3857)
    evac_points_3857 = evac_points_gdf.to_crs(epsg=3857)
    
    # ============================================================
    # Figura
    # ============================================================
    
    fig, ax = plt.subplots(figsize=FIG_SIZE)
    
    # ============================================================
    # Extensión del mapa
    # ============================================================
    
    # límites basados en los nodos, no en toda la geometría comunal
    xmin, ymin, xmax, ymax = nodes_3857.total_bounds
    
    # padding en metros (ajústalo a gusto)
    padding_x = 3500
    padding_y = 3500
    
    ax.set_xlim(-8.151e6, -8.116e6)
    ax.set_ylim(-4.420e6, -4.390e6)
    
    # ============================================================
    # Límites comunales, estilo GeoPandas gallery
    # ============================================================
    # Esto es lo que genera el efecto de "áreas azules delineadas".
    
    comunas_3857.plot(
        ax=ax,
        color="tab:blue",
        alpha=0.32,
        edgecolor="black",
        linewidth=0.8,
        zorder=2,
        label="Communal boundaries"
    )
    
    # ============================================================
    # Mapa base
    # ============================================================
    # Primero agregamos el fondo sin etiquetas.
    # Luego agregamos solo las etiquetas encima, como sugiere la galería.
    
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.PositronNoLabels,
        zoom=MAP_ZOOM_BASE,
        reset_extent=False,
        attribution_size=5
    )
    
    # Volvemos a dibujar los bordes comunales encima del basemap,
    # para asegurar que queden visibles.
    comunas_3857.boundary.plot(
        ax=ax,
        color="black",
        linewidth=0.8,
        zorder=3
    )
    
    # Relleno comunal translúcido encima.
    comunas_3857.plot(
        ax=ax,
        color="tab:blue",
        alpha=0.25,
        edgecolor="black",
        linewidth=0.8,
        zorder=4,
        label="Communal boundaries"
    )
    
    # ============================================================
    # Carta de inundación por tsunami
    # ============================================================
    
    tsunami_area_3857.plot(
        ax=ax,
        color="red",
        alpha=0.28,
        edgecolor="red",
        linewidth=1.0,
        zorder=5,
        label="Tsunami inundation area"
    )
    
    # Borde más marcado de la zona de inundación
    tsunami_area_3857.boundary.plot(
        ax=ax,
        color="red",
        linewidth=1.1,
        alpha=0.30,
        zorder=6
    )
    
    # ============================================================
    # Puntos de evacuación
    # ============================================================
    
    evac_points_3857.plot(
        ax=ax,
        markersize=15,
        marker="o",
        color="limegreen",
        edgecolor="darkgreen",
        linewidth=0.6,
        alpha=0.50,
        zorder=7,
        label="Evacuation points"
    )
    
    # ============================================================
    # Etiquetas de comunas
    # ============================================================
    
    for _, row in comunas_3857.iterrows():
        point = row.geometry.representative_point()
        label = row[comuna_col]
    
        text = ax.annotate(
            label,
            xy=(point.x, point.y),
            ha="center",
            va="center",
            fontsize=7,
            color="dimgray",
            zorder=9
        )
    
        text.set_path_effects([
            path_effects.Stroke(linewidth=2.5, foreground="white"),
            path_effects.Normal()
        ])
    
    # ============================================================
    # Nodos TTAP
    # ============================================================
    
    nodes_3857[nodes_3857["type"] == "Demand node"].plot(
        ax=ax,
        markersize=90,
        marker="o",
        color="white",
        edgecolor="black",
        linewidth=1.0,
        alpha=0.90,
        zorder=8,
        label="Demand nodes"
    )
    
    nodes_3857[nodes_3857["type"] == "Base"].plot(
        ax=ax,
        markersize=210,
        marker="*",
        color="darkred",
        edgecolor="black",
        linewidth=0.9,
        alpha=0.98,
        zorder=8,
        label="Base"
    )
    
    # ============================================================
    # Etiquetas de nodos
    # ============================================================
    
    label_offsets = {
        "Base": (9, 9),
        "A": (8, 6),
        "B": (8, 6),
        "C": (8, 6),
        "D": (8, 6),
        "E": (8, 6),
        "F": (8, 6),
        "G": (8, 6),
    }
    
    for _, row in nodes_3857.iterrows():
    
        dx, dy = label_offsets.get(row["name"], (8, 6))
    
        if row["name"] == "Base":
            label_color = "darkred"
            font_size = 12
        else:
            label_color = "black"
            font_size = 11
    
        text = ax.annotate(
            row["name"],
            xy=(row.geometry.x, row.geometry.y),
            xytext=(dx, dy),
            textcoords="offset points",
            color=label_color,
            fontweight="bold",
            fontsize=font_size,
            zorder=8
        )
    
        text.set_path_effects([
            path_effects.Stroke(linewidth=2.8, foreground="white"),
            path_effects.Normal()
        ])
    
    # ============================================================
    # Estética final
    # ============================================================
    
    ax.set_title(
        "TTAP demand nodes and tsunami inundation area",
        fontsize=12,
        fontweight="bold",
        pad=12
    )
    
    # Quitar valores de los ejes
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Quitar marcas de los ejes
    ax.tick_params(
        left=False,
        bottom=False,
        labelleft=False,
        labelbottom=False
    )
    
    plt.tight_layout()
    
    # ============================================================
    # Guardado
    # ============================================================
    
    png_path = os.path.join(FIGURES_DIR, f"{MAP_NAME_TSUNAMI}.png")
    pdf_path = os.path.join(FIGURES_DIR, f"{MAP_NAME_TSUNAMI}.pdf")
    
    fig.savefig(
        png_path,
        dpi=1000,
        bbox_inches="tight",
        facecolor="white"
    )
    
    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )
    
    print(f"Map saved at: {png_path}")
    print(f"Map saved at: {pdf_path}")
    
    plt.show()


### 4. Políticas Greedy Benefit y Greedy Density

Se ejecutan sobre copias nuevas del mismo escenario. Density usa
`benefit / (flight_time_to_task + SERVICE_TIME)`, igual que en A/B.
El retorno y recuperación se verifican para factibilidad; no forman parte de
ese denominador. Los desempates usan beneficio, finalización y Task ID.


In [ ]:
class GreedyNode:
    def __init__(self, name, lat, lon):
        self.name = name
        self.lat = lat
        self.lon = lon
        self.tasks = []

    def add_task(self, task):
        self.tasks.append(task)

    def distance_to(self, other):
        """
        Distancia GPS aproximada entre nodos en kilómetros.
        """

        radius_earth_km = 6371.0088

        lat1_rad = math.radians(self.lat)
        lon1_rad = math.radians(self.lon)
        lat2_rad = math.radians(other.lat)
        lon2_rad = math.radians(other.lon)

        delta_lat = lat2_rad - lat1_rad
        delta_lon = lon2_rad - lon1_rad

        a = (
            math.sin(delta_lat / 2.0) ** 2
            + math.cos(lat1_rad)
            * math.cos(lat2_rad)
            * math.sin(delta_lon / 2.0) ** 2
        )

        c = 2.0 * math.asin(math.sqrt(a))

        return radius_earth_km * c


class GreedyTask:
    def __init__(
        self,
        task_id,
        visual_search,
        material_transport,
        medical_assistance,
        personnel_transfer,
        optimal_time,
        effective_time,
        ineffective_time,
    ):
        self.task_id = task_id
        self.visual_search = visual_search
        self.material_transport = material_transport
        self.medical_assistance = medical_assistance
        self.personnel_transfer = personnel_transfer
        self.optimal_time = optimal_time
        self.effective_time = effective_time
        self.ineffective_time = ineffective_time
        self.is_completed = False
        self.completion_time = None

    def calculate_satisfaction(self, t_finish):
        return satisfaction(
            t_finish,
            tasks[self.task_id]
        )


    def calculate_benefit(self, t_finish):
        return task_benefit(
            self.task_id,
            t_finish
        )


class GreedyHelicopter:
    def __init__(
        self,
        hid,
        visual_search_capability,
        load_capacity,
        medical_capacity,
        passenger_limit,
        speed_kmh,
        position,
    ):
        self.helicopter_id = hid
        self.visual_search_capability = visual_search_capability

        self.initial_load_capacity = load_capacity
        self.initial_medical_capacity = medical_capacity
        self.initial_passenger_limit = passenger_limit

        self.load_capacity = load_capacity
        self.medical_capacity = medical_capacity
        self.passenger_limit = passenger_limit

        self.speed_kmh = speed_kmh
        self.position = position
        self.available_time = 0.0
        self.blocked = False

    def can_complete_task(self, task):
        return (
            (not task.visual_search or self.visual_search_capability)
            and self.load_capacity >= task.material_transport
            and self.medical_capacity >= task.medical_assistance
            and self.passenger_limit >= task.personnel_transfer
        )

    def execute_task(self, task, t_finish):
        self.load_capacity -= task.material_transport
        self.medical_capacity -= task.medical_assistance
        self.passenger_limit -= task.personnel_transfer

        task.is_completed = True
        task.completion_time = t_finish

        return task.calculate_benefit(t_finish)

    def return_to_base_and_recover(self, base_node):
        tau = self.position.distance_to(base_node) / self.speed_kmh * 60.0
        duration = tau + RECOVERY_TIME
    
        k_now = int(round(self.available_time / TIME_UNIT))
        k_finish = k_now + ceil_steps(duration, TIME_UNIT)
        
        self.available_time = time_from_step(k_finish, TIME_UNIT)
        self.position = base_node
        
        self.load_capacity = self.initial_load_capacity
        self.medical_capacity = self.initial_medical_capacity
        self.passenger_limit = self.initial_passenger_limit
        
        return tau, RECOVERY_TIME


def initialize_greedy_graph():
    node_map = {
        node_name: GreedyNode(
            node_name,
            coord[0],
            coord[1],
        )
        for node_name, coord in nodes.items()
    }

    base = node_map["Base"]

    greedy_tasks = {}
    for i, t in tasks.items():
        greedy_tasks[i] = GreedyTask(
            task_id=i,
            visual_search=t["visual"],
            material_transport=t["load"],
            medical_assistance=t["med"],
            personnel_transfer=t["pers"],
            optimal_time=t["t_opt"],
            effective_time=t["t_eff"],
            ineffective_time=t["t_ineff"],
        )

    for task_id, node_name in task_to_node.items():
        node_map[node_name].add_task(greedy_tasks[task_id])

    greedy_helos = []
    for h, p in helicopters.items():
        greedy_helos.append(
            GreedyHelicopter(
                hid=helo_to_int(h),
                visual_search_capability=p["visual"],
                load_capacity=p["load"],
                medical_capacity=p["med"],
                passenger_limit=p["pers"],
                speed_kmh=p["speed"],
                position=base,
            )
        )

    return list(node_map.values()), greedy_helos, greedy_tasks


def next_available_helo(helicopters_list):
    active = [h for h in helicopters_list if not h.blocked]
    if not active:
        return None
    return min(active, key=lambda h: (h.available_time, h.helicopter_id))


def solve_greedy_ttap(T_max=TMAX, verbose=True, rule="benefit"):
    """Fresh construction: benefit, or benefit/(flight to task + service)."""
    if rule not in {"benefit", "density_operational"}:
        raise ValueError("rule must be benefit or density_operational")
    method = "Greedy Benefit" if rule == "benefit" else "Greedy Density"
    t0 = time.perf_counter()
    greedy_nodes, greedy_helos, greedy_tasks = initialize_greedy_graph()
    node_map = {node.name: node for node in greedy_nodes}
    base = node_map["Base"]
    pending = set(greedy_tasks)
    rows = []

    def recover(helo):
        start_node, start = helo.position.name, helo.available_time
        flight = helo.position.distance_to(base) / helo.speed_kmh * 60.0
        finish = time_from_step(int(round(start / TIME_UNIT))
                               + ceil_steps(flight + RECOVERY_TIME, TIME_UNIT), TIME_UNIT)
        if finish > T_max + 1e-9:
            raise RuntimeError(method + ": mandatory return exceeds the mission horizon")
        helo.return_to_base_and_recover(base)
        rows.append({"Method": method, "Helo ID": helo.helicopter_id,
                     "Task ID": "Idle", "Start Node": start_node, "Node": "Base",
                     "Start Time": start, "Flight Time": flight, "Service Time": RECOVERY_TIME,
                     "Total Task Time": flight + RECOVERY_TIME, "Completion Time": helo.available_time,
                     "Benefit": 0., "Selection Score": 0., "Remaining Load": helo.load_capacity,
                     "Remaining Medical": helo.medical_capacity,
                     "Remaining Personnel": helo.passenger_limit, "Status": "Return/Recovery"})

    while pending:
        helo = next_available_helo(greedy_helos)
        if helo is None:
            break
        if helo.available_time >= T_max:
            helo.blocked = True
            continue
        start = helo.available_time
        k_now = int(round(start / TIME_UNIT))
        candidates = []
        for task_id in sorted(pending):
            task = greedy_tasks[task_id]
            if not helo.can_complete_task(task):
                continue
            node = node_map[task_to_node[task_id]]
            flight = helo.position.distance_to(node) / helo.speed_kmh * 60.0
            duration = flight + SERVICE_TIME
            finish = time_from_step(k_now + ceil_steps(duration, TIME_UNIT), TIME_UNIT)
            return_flight = node.distance_to(base) / helo.speed_kmh * 60.0
            recovery_finish = time_from_step(int(round(finish / TIME_UNIT))
                                             + ceil_steps(return_flight + RECOVERY_TIME, TIME_UNIT), TIME_UNIT)
            if finish > T_max or recovery_finish > T_max:
                continue
            benefit = task.calculate_benefit(finish)
            if benefit <= ZERO_BENEFIT_TOL:
                continue
            score = benefit if rule == "benefit" else benefit / duration
            candidates.append((score, benefit, finish, task_id, flight, duration))
        if not candidates:
            if helo.position is base:
                helo.blocked = True
            else:
                recover(helo)
            continue
        score, benefit, finish, task_id, flight, duration = min(
            candidates, key=lambda x: (-x[0], -x[1], x[2], x[3]))
        task = greedy_tasks[task_id]
        start_node = helo.position.name
        helo.position = node_map[task_to_node[task_id]]
        helo.available_time = finish
        helo.execute_task(task, finish)
        pending.remove(task_id)
        rows.append({"Method": method, "Helo ID": helo.helicopter_id, "Task ID": task_id,
                     "Start Node": start_node, "Node": helo.position.name, "Start Time": start,
                     "Flight Time": flight, "Service Time": SERVICE_TIME, "Total Task Time": duration,
                     "Completion Time": finish, "Benefit": benefit, "Selection Score": score,
                     "Remaining Load": helo.load_capacity, "Remaining Medical": helo.medical_capacity,
                     "Remaining Personnel": helo.passenger_limit, "Status": "Task Executed",
                     "Priority Class": tasks[task_id]["priority_class"],
                     "Task Weight": tasks[task_id]["weight"],
                     "Temporal Satisfaction": task.calculate_satisfaction(finish)})
        if verbose:
            print(f"{method}: H{helo.helicopter_id} -> T{task_id}, finish={finish:.1f}, benefit={benefit:.6f}")

    for helo in greedy_helos:
        if helo.position is not base:
            recover(helo)
    columns = ["Method", "Helo ID", "Task ID", "Start Node", "Node", "Start Time", "Flight Time",
               "Service Time", "Total Task Time", "Completion Time", "Benefit", "Selection Score",
               "Remaining Load", "Remaining Medical", "Remaining Personnel", "Status",
               "Priority Class", "Task Weight", "Temporal Satisfaction"]
    frame = pd.DataFrame(rows, columns=columns).sort_values(
        ["Completion Time", "Helo ID"], kind="stable").reset_index(drop=True)
    executed = frame[frame["Status"].eq("Task Executed")]
    completed = executed["Task ID"].astype(int).tolist()
    summary = {"method": method, "benefit": float(executed["Benefit"].sum()),
               "completed_tasks": len(completed), "uncompleted_tasks": sorted(set(tasks) - set(completed)),
               "runtime_sec": time.perf_counter() - t0, "optimal": False,
               "has_incumbent": True, "solver_message": "Constructive heuristic", "mip_gap": None,
               "rule": rule, "objective_mode": OBJECTIVE_MODE}
    validate_route_schedule(frame, method)
    return frame, summary, executed["Completion Time"].tolist(), executed["Benefit"].tolist()


def validate_route_schedule(frame, method):
    """Check both policies and the decoded MILP against the same mission rules."""
    if frame.empty:
        return True
    executed = frame[frame["Status"].eq("Task Executed")]
    if executed["Task ID"].duplicated().any():
        raise AssertionError(method + ": a task was executed twice")
    for helo_id, group in frame.groupby("Helo ID"):
        h = helo_to_str(helo_id)
        position, clock = "Base", 0.
        remaining = {key: helicopters[h][key] for key in ("load", "med", "pers")}
        for _, row in group.sort_values("Completion Time", kind="stable").iterrows():
            flight = travel_time_minutes(h, position, row["Node"])
            finish = time_from_step(int(round(clock / TIME_UNIT))
                                    + ceil_steps(flight + row["Service Time"], TIME_UNIT), TIME_UNIT)
            assert np.isclose(row["Flight Time"], flight, atol=1e-8), method
            assert np.isclose(row["Completion Time"], finish, atol=1e-8), method
            assert finish <= TMAX + 1e-8, method
            if row["Status"] == "Task Executed":
                i = int(row["Task ID"])
                assert task_to_node[i] == row["Node"], method
                assert can_execute_task_by_resources(h, i, remaining["load"], remaining["med"], remaining["pers"]), method
                assert np.isclose(row["Service Time"], SERVICE_TIME), method
                assert np.isclose(row["Benefit"], task_benefit(i, finish), atol=1e-8), method
                for key in remaining:
                    remaining[key] -= tasks[i][key]
            else:
                assert row["Node"] == "Base" and np.isclose(row["Service Time"], RECOVERY_TIME), method
                assert abs(row["Benefit"]) < 1e-10, method
                remaining = {key: helicopters[h][key] for key in remaining}
            for key, column in [("load", "Remaining Load"), ("med", "Remaining Medical"), ("pers", "Remaining Personnel")]:
                assert remaining[key] >= 0 and np.isclose(row[column], remaining[key]), method
            position, clock = row["Node"], finish
        assert position == "Base", method + ": route does not finish at Base"
    return True


df_greedy_benefit, greedy_benefit_summary, _, _ = solve_greedy_ttap(rule="benefit", verbose=False)
df_greedy_density, greedy_density_summary, _, _ = solve_greedy_ttap(rule="density_operational", verbose=False)

# Compatibility names refer explicitly to Benefit; comparison cells use both.
df_greedy, greedy_summary = df_greedy_benefit, greedy_benefit_summary
GREEDY_RESULTS = {"Greedy Benefit": (df_greedy_benefit, greedy_benefit_summary),
                  "Greedy Density": (df_greedy_density, greedy_density_summary)}
display(pd.DataFrame([greedy_benefit_summary, greedy_density_summary]))


### 5. Método Exacto TTAP (Solver:HIGHS - High-performance Software for Linear Optimization)

In [ ]:
def ceil_steps(minutes, time_unit):
    return int(math.ceil((minutes - 1e-9) / time_unit))


def time_from_step(k, time_unit):
    return k * time_unit


def make_state(node, k, load, med, pers):
    return (node, int(k), int(load), int(med), int(pers))


def start_state(h):
    p = helicopters[h]
    return make_state("Base", 0, p["load"], p["med"], p["pers"])


def sink_state():
    return ("SINK", -1, 0, 0, 0)


def build_state_graph_for_helicopter(h, time_unit=TIME_UNIT):
    tmax_steps = int(round(TMAX / time_unit))

    s0 = start_state(h)
    sink = sink_state()

    states = set([s0])
    queue = deque([s0])
    arcs = []

    while queue:
        current_state = queue.popleft()

        node, k, load, med, pers = current_state
        current_time = time_from_step(k, time_unit)

        # -----------------------------
        # Arcos de tarea
        # -----------------------------
        for task_id in tasks:

            if not can_execute_task_by_resources(h, task_id, load, med, pers):
                continue

            task_node = task_to_node[task_id]

            flight = travel_time_minutes(h, node, task_node)
            duration = flight + SERVICE_TIME
            duration_steps = ceil_steps(duration, time_unit)

            k_new = k + duration_steps

            if k_new > tmax_steps:
                continue

            finish_time = time_from_step(k_new, time_unit)

            task = tasks[task_id]

            new_state = make_state(
                task_node,
                k_new,
                load - task["load"],
                med - task["med"],
                pers - task["pers"],
            )

            benefit = task_benefit(task_id, finish_time)

            # Evita crear arcos de tareas que ya no generan valor operacional.
            if benefit <= ZERO_BENEFIT_TOL:
                continue

            arcs.append({
                "h": h,
                "from": current_state,
                "to": new_state,
                "kind": "task",
                "task": task_id,
                "start_time": current_time,
                "finish_time": finish_time,
                "flight_time": flight,
                "service_time": SERVICE_TIME,
                "benefit": benefit,
                "satisfaction": satisfaction(finish_time,task),
                "priority_class": task["priority_class"],
                "task_weight": task["weight"],
            })

            if new_state not in states:
                states.add(new_state)
                queue.append(new_state)

        # -----------------------------
        # Arco de retorno a base
        # -----------------------------
        if node != "Base":
            flight = travel_time_minutes(h, node, "Base")
            duration = flight + RECOVERY_TIME
            duration_steps = ceil_steps(duration, time_unit)

            k_new = k + duration_steps

            if k_new <= tmax_steps:
                p = helicopters[h]

                new_state = make_state(
                    "Base",
                    k_new,
                    p["load"],
                    p["med"],
                    p["pers"],
                )

                arcs.append({
                    "h": h,
                    "from": current_state,
                    "to": new_state,
                    "kind": "return",
                    "task": None,
                    "start_time": current_time,
                    "finish_time": time_from_step(k_new, time_unit),
                    "flight_time": flight,
                    "service_time": RECOVERY_TIME,
                    "benefit": 0.0,
                    "satisfaction": None,
                    "priority_class": None,
                    "task_weight": None,
                })

                if new_state not in states:
                    states.add(new_state)
                    queue.append(new_state)

    # -----------------------------
    # Arcos de término
    # -----------------------------
    states.add(sink)

    for s in list(states):
        if s == sink:
            continue

        node, k, load, med, pers = s

        if node != "Base":
            continue

        arcs.append({
            "h": h,
            "from": s,
            "to": sink,
            "kind": "end",
            "task": None,
            "start_time": time_from_step(k, time_unit),
            "finish_time": time_from_step(k, time_unit),
            "flight_time": 0.0,
            "service_time": 0.0,
            "benefit": 0.0,
            "satisfaction": None,
            "priority_class": None,
            "task_weight": None,
        })

    return list(states), arcs


def build_full_exact_graph(time_unit=TIME_UNIT):
    states_by_h = {}
    arcs_by_h = {}
    all_arcs = []

    print("\nConstruyendo red de estados exacta...\n")

    for h in helicopters:
        states, arcs = build_state_graph_for_helicopter(h, time_unit)

        states_by_h[h] = states
        arcs_by_h[h] = arcs
        all_arcs.extend(arcs)

        print(f"{h}: {len(states):,} estados | {len(arcs):,} arcos")

    print(f"\nTotal arcos/variables binarias: {len(all_arcs):,}")

    return states_by_h, arcs_by_h, all_arcs


def build_exact_milp_matrices(states_by_h, arcs_by_h, all_arcs):
    n_vars = len(all_arcs)

    c = np.array([-arc["benefit"] for arc in all_arcs], dtype=float)

    lb = np.zeros(n_vars)
    ub = np.ones(n_vars)
    integrality = np.ones(n_vars, dtype=int)

    row_idx = []
    col_idx = []
    vals = []

    rhs_lb = []
    rhs_ub = []

    current_row = 0

    arc_global_index = {}
    offset = 0

    for h in helicopters:
        for local_idx, arc in enumerate(arcs_by_h[h]):
            arc_global_index[(h, local_idx)] = offset + local_idx
        offset += len(arcs_by_h[h])

    # -----------------------------
    # Conservación de flujo
    # -----------------------------
    for h in helicopters:
        outgoing = defaultdict(list)
        incoming = defaultdict(list)

        for local_idx, arc in enumerate(arcs_by_h[h]):
            j = arc_global_index[(h, local_idx)]
            outgoing[arc["from"]].append(j)
            incoming[arc["to"]].append(j)

        s0 = start_state(h)
        sink = sink_state()

        for state in states_by_h[h]:
            for j in outgoing.get(state, []):
                row_idx.append(current_row)
                col_idx.append(j)
                vals.append(1.0)

            for j in incoming.get(state, []):
                row_idx.append(current_row)
                col_idx.append(j)
                vals.append(-1.0)

            if state == s0:
                rhs = 1.0
            elif state == sink:
                rhs = -1.0
            else:
                rhs = 0.0

            rhs_lb.append(rhs)
            rhs_ub.append(rhs)
            current_row += 1

    # -----------------------------
    # Cada tarea máximo una vez
    # -----------------------------
    for task_id in tasks:
        for j, arc in enumerate(all_arcs):
            if arc["kind"] == "task" and arc["task"] == task_id:
                row_idx.append(current_row)
                col_idx.append(j)
                vals.append(1.0)

        rhs_lb.append(-np.inf)
        rhs_ub.append(1.0)
        current_row += 1

    A = coo_matrix(
        (vals, (row_idx, col_idx)),
        shape=(current_row, n_vars)
    ).tocsr()

    constraints = LinearConstraint(
        A,
        np.array(rhs_lb, dtype=float),
        np.array(rhs_ub, dtype=float)
    )

    bounds = Bounds(lb, ub)

    return c, integrality, bounds, constraints, A


def solve_exact_ttap_v01(time_unit=TIME_UNIT, time_limit=180, mip_rel_gap=1e-6, verbose=True):
    t0 = time.time()

    states_by_h, arcs_by_h, all_arcs = build_full_exact_graph(time_unit)

    c, integrality, bounds, constraints, A = build_exact_milp_matrices(
        states_by_h,
        arcs_by_h,
        all_arcs
    )

    print("\nResolviendo MILP exacto...\n")
    print(f"Variables binarias : {len(all_arcs):,}")
    print(f"Restricciones      : {A.shape[0]:,}")
    print(f"No ceros matriz A  : {A.nnz:,}")
    print(f"Time limit         : {time_limit} segundos")
    print(f"MIP relative gap   : {mip_rel_gap}")

    solver_started = time.perf_counter()
    result = milp(
        c=c,
        integrality=integrality,
        bounds=bounds,
        constraints=constraints,
        options={
            "time_limit": time_limit,
            "mip_rel_gap": mip_rel_gap,
            "disp": verbose,
        }
    )

    solver_seconds = time.perf_counter() - solver_started
    runtime = time.time() - t0

    has_incumbent = result.x is not None
    objective_value = -float(result.fun) if has_incumbent else np.nan
    mip_gap = getattr(result, 'mip_gap', np.nan)
    if has_incumbent:
        activity = A @ result.x
        assert np.all(np.abs(result.x-np.rint(result.x)) <= 1e-5)
        assert np.all(result.x >= -1e-6) and np.all(result.x <= 1+1e-6)
        assert np.all(activity >= constraints.lb-1e-5)
        assert np.all(activity <= constraints.ub+1e-5)

    summary = {
        "method": "Exact-HiGHS" if result.success else "HiGHS incumbent" if has_incumbent else "HiGHS (no incumbent)",
        "benefit": objective_value,
        "completed_tasks": None,
        "uncompleted_tasks": None,
        "runtime_sec": runtime,
        "optimal": bool(result.success),
        "has_incumbent": has_incumbent,
        "solver_sec": solver_seconds,
        "objective_mode": OBJECTIVE_MODE,
        "solver_message": result.message,
        "mip_gap": mip_gap,
        "variables": len(all_arcs),
        "constraints": A.shape[0],
        "nonzeros": A.nnz,
    }

    print("\n============================================================")
    print("EXACT RESULTS")
    print("============================================================")
    print(f"Solver status      : {result.message}")
    print(f"Beneficio total    : {objective_value:.6f}")
    print(f"Runtime            : {runtime:.6f} sec")
    print(f"Óptimo certificado : {result.success}")

    return result, all_arcs, states_by_h, arcs_by_h, summary


def decode_exact_solution(result, all_arcs):
    selected_indices = [
        j for j, value in enumerate(result.x)
        if value > 0.5
    ]

    selected_arcs = [all_arcs[j] for j in selected_indices]

    selected_by_h = defaultdict(list)

    for arc in selected_arcs:
        selected_by_h[arc["h"]].append(arc)

    rows = []

    for h in helicopters:
        out_arc = {
            arc["from"]: arc
            for arc in selected_by_h[h]
        }

        current = start_state(h)

        while True:
            arc = out_arc.get(current)

            if arc is None:
                break

            if arc["kind"] == "end":
                break

            if arc["kind"] == "task":
                task_id = arc["task"]
                next_state = arc["to"]

                rows.append({
                    "Method": "Exact-HiGHS",
                    "Helo ID": helo_to_int(h),
                    "Task ID": task_id,
                    "Node": task_to_node[task_id],
                    "Start Node": current[0],
                    "Start Time": arc["start_time"],
                    "Flight Time": arc["flight_time"],
                    "Service Time": arc["service_time"],
                    "Total Task Time": arc["flight_time"] + arc["service_time"],
                    "Completion Time": arc["finish_time"],
                    "Benefit": arc["benefit"],
                    "Remaining Load": next_state[2],
                    "Remaining Medical": next_state[3],
                    "Remaining Personnel": next_state[4],
                    "Status": "Task Executed",
                    "Priority Class": arc["priority_class"],
                    "Task Weight": arc["task_weight"],
                    "Temporal Satisfaction": arc["satisfaction"],
                })

            elif arc["kind"] == "return":
                next_state = arc["to"]

                rows.append({
                    "Method": "Exact-HiGHS",
                    "Helo ID": helo_to_int(h),
                    "Task ID": "Idle",
                    "Node": "Base",
                    "Start Node": current[0],
                    "Start Time": arc["start_time"],
                    "Flight Time": arc["flight_time"],
                    "Service Time": arc["service_time"],
                    "Total Task Time": arc["flight_time"] + arc["service_time"],
                    "Completion Time": arc["finish_time"],
                    "Benefit": 0.0,
                    "Remaining Load": next_state[2],
                    "Remaining Medical": next_state[3],
                    "Remaining Personnel": next_state[4],
                    "Status": "Idle",
                    "Priority Class": arc["priority_class"],
                    "Task Weight": arc["task_weight"],
                    "Temporal Satisfaction": arc["satisfaction"],
                })

            current = arc["to"]

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values(
            by=["Completion Time", "Helo ID"]
        ).reset_index(drop=True)

    return df


result_exact, all_arcs, states_by_h, arcs_by_h, exact_summary = solve_exact_ttap_v01(
    time_unit=TIME_UNIT, time_limit=EXACT_TIME_LIMIT,
    mip_rel_gap=EXACT_MIP_REL_GAP, verbose=True)
if exact_summary['has_incumbent']:
    df_exact=decode_exact_solution(result_exact,all_arcs)
    if df_exact.empty:
        df_exact=df_greedy_benefit.iloc[:0].copy()
    df_exact['Method']=exact_summary['method']
    validate_route_schedule(df_exact,exact_summary['method'])
    decoded=float(df_exact.loc[df_exact['Status'].eq('Task Executed'),'Benefit'].sum())
    assert np.isclose(decoded,exact_summary['benefit'],atol=1e-6)
    completed=sorted(df_exact.loc[df_exact['Status'].eq('Task Executed'),'Task ID'].astype(int))
    exact_summary['completed_tasks']=len(completed)
    exact_summary['uncompleted_tasks']=sorted(set(tasks)-set(completed))
else:
    df_exact=df_greedy_benefit.iloc[:0].copy()
    exact_summary['completed_tasks']=np.nan
    exact_summary['uncompleted_tasks']=None
METHOD_RESULTS={**GREEDY_RESULTS,exact_summary['method']:(df_exact,exact_summary)}
for method,(frame,summary) in METHOD_RESULTS.items():
    if summary['has_incumbent']:
        decoded=float(frame.loc[frame['Status'].eq('Task Executed'),'Benefit'].sum())
        assert np.isclose(decoded,summary['benefit'],atol=1e-6)
display(df_exact)


#### 5.1. Detalle de States y Arcs:

In [ ]:
def states_to_dataframe(states_by_h, time_unit=TIME_UNIT):
    rows = []

    for h, states in states_by_h.items():
        for state in states:
            node, k, load, med, pers = state

            if node == "SINK":
                continue

            rows.append({
                "Helo": h,
                "Node": node,
                "Time Step": k,
                "Time": k * time_unit,
                "Remaining Load": load,
                "Remaining Medical": med,
                "Remaining Personnel": pers,
                "State": state,
            })

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["Helo", "Time Step", "Node", "Remaining Load", "Remaining Medical", "Remaining Personnel"]
        )
        .reset_index(drop=True)
    )

In [ ]:
def arcs_to_dataframe(arcs_by_h):
    rows = []

    for h, arcs in arcs_by_h.items():
        for arc_id, arc in enumerate(arcs):
            rows.append({
                "Helo": h,
                "Arc ID": arc_id,
                "Kind": arc["kind"],
                "Task": arc["task"],
                "From State": arc["from"],
                "To State": arc["to"],
                "Start Time": arc["start_time"],
                "Finish Time": arc["finish_time"],
                "Flight Time": arc["flight_time"],
                "Service Time": arc["service_time"],
                "Benefit": arc["benefit"],
            })

    return (
        pd.DataFrame(rows)
        .sort_values(["Helo", "Start Time", "Finish Time", "Kind", "Arc ID"])
        .reset_index(drop=True)
    )

##### States:

In [ ]:
df_states_exact = states_to_dataframe(states_by_h)
display(df_states_exact[df_states_exact["Helo"] == "H6"].head(10))

In [ ]:
len(df_states_exact[df_states_exact["Helo"] == "H2"])

##### Arcs:

In [ ]:
df_arcs_exact = arcs_to_dataframe(arcs_by_h)
display(df_arcs_exact[df_arcs_exact["Helo"] == "H5"].head(5))

### 6. Resumen comparativo básico:

In [ ]:
certified_reference = (exact_summary['benefit'] if exact_summary['optimal']
                       and exact_summary['has_incumbent'] and exact_summary['benefit'] > 0 else None)
comparison_rows=[]
for method,(frame,summary) in METHOD_RESULTS.items():
    value=summary['benefit']
    if certified_reference is not None and summary['has_incumbent']:
        if value > certified_reference + 1e-6:
            raise AssertionError('A feasible value exceeds the certified reference')
        shortfall=100*max(0.,certified_reference-value)/certified_reference
    else:
        shortfall=np.nan
    comparison_rows.append({'Method':method,'Objective Mode':OBJECTIVE_MODE,
                            'Accumulated Benefit':value,'Completed Tasks':summary['completed_tasks'],
                            'Completion Rate':summary['completed_tasks']/len(tasks),
                            'Uncompleted Tasks':summary['uncompleted_tasks'],
                            'Runtime sec':summary['runtime_sec'],'Has Incumbent':summary['has_incumbent'],
                            'Optimal Certified':summary['optimal'],'MIP Gap':summary['mip_gap'],
                            'Shortfall to Certified Optimum %':shortfall})
basic_comparison=pd.DataFrame(comparison_rows)
basic_comparison.to_csv(Path(FIGURES_DIR)/'talcahuano_comparison.csv',index=False)
df_routes_all=pd.concat([frame.assign(Method=method) for method,(frame,_) in METHOD_RESULTS.items()],ignore_index=True)
df_routes_all.to_csv(Path(FIGURES_DIR)/'talcahuano_routes.csv',index=False)
manifest={'source_commit':'0be00802518836ed652035dad6d504cfbb614c8a',
          'source_blob':'eac2e8b11e3436a2c0e378256a8db5cb6cfc5578',
          'scenario':SCENARIO_NAME,'objective_mode':OBJECTIVE_MODE,
          'priority_weights':PRIORITY_WEIGHTS,'impact_references':IMPACT_REFERENCES,
          'exact_solver_allowance_sec':EXACT_TIME_LIMIT,'mip_relative_tolerance':EXACT_MIP_REL_GAP,
          'time_unit_minutes':TIME_UNIT,'mission_horizon_minutes':TMAX,'png_dpi':PNG_DPI,
          'density_denominator':'flight to task plus service, continuous minutes',
          'python':sys.version.split()[0],'numpy':np.__version__,'pandas':pd.__version__,
          'scipy':scipy.__version__,'matplotlib':matplotlib.__version__,
          'certified_reference_value':certified_reference}
(Path(FIGURES_DIR)/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
display(basic_comparison)

def latex_number(value,digits=4):
    return '--' if pd.isna(value) else f'{value:.{digits}f}'

tex=[r'\begin{table}[!htbp]',r'\centering',
     r'\caption{Talcahuano: comparison on the fixed GPS scenario.}',r'\label{tab:gps-three-methods}',
     r'\scriptsize',r'\setlength{\tabcolsep}{3pt}',r'\renewcommand{\arraystretch}{1.08}',
     r'\begin{tabular}{@{}lrrrrc@{}}',r'\toprule',
     r'Method & Benefit & Tasks & \makecell{Runtime\\(s)} & \makecell{Shortfall\\(\%)} & \makecell{Optimal\\certified} \\',
     r'\midrule']
for _,row in basic_comparison.iterrows():
    method=row['Method'].replace('Greedy Benefit',r'\makecell[l]{Greedy\\Benefit}').replace(
        'Greedy Density',r'\makecell[l]{Greedy\\Density}').replace('Exact-HiGHS',r'\makecell[l]{Exact-\\HiGHS}')
    certificate=('Yes' if row['Optimal Certified'] else 'No') if method not in {
        r'\makecell[l]{Greedy\\Benefit}',r'\makecell[l]{Greedy\\Density}'} else '--'
    tex.append(' & '.join([method,latex_number(row['Accumulated Benefit']),
                          latex_number(row['Completed Tasks'],0),latex_number(row['Runtime sec']),
                          latex_number(row['Shortfall to Certified Optimum %'],2),certificate])+r' \\')
tex += [r'\bottomrule',r'\end{tabular}',r'\begin{minipage}{0.98\columnwidth}',r'\footnotesize',
        'All methods use the same fixed scenario and '+OBJECTIVE_MODE+' objective. '
        'There are no repeated-instance means or standard deviations. Runtime includes construction and solving '
        'for the MILP. Shortfalls use a certified optimum only; otherwise they are unavailable. '
        r'Certification uses relative MIP tolerance $10^{-6}$.',r'\end{minipage}',r'\end{table}']
(Path(FIGURES_DIR)/'Table_Talcahuano_three_methods.tex').write_text('\n'.join(tex)+'\n')
if certified_reference is None:
    print('No certified optimum: greedy-to-optimum shortfalls are not reported.')


### 7. Gantt Chart:

In [ ]:
def common_mission_xlim():
    maximum=max((float(frame['Completion Time'].max()) for frame,_ in METHOD_RESULTS.values()
                 if not frame.empty),default=0.)
    return (0.,min(TMAX,max(10.,10*math.ceil((maximum+5.)/10))))


def draw_schedule(ax,frame,method,small=False):
    colors={'travel':'#9ecae1','service':'#3182bd','recovery':'#cccccc'}
    previous={}
    for _,row in frame.sort_values(['Completion Time','Helo ID'],kind='stable').iterrows():
        h=helo_to_int(row['Helo ID'])
        start=float(row['Start Time']) if 'Start Time' in row and pd.notna(row['Start Time']) else previous.get(h,0.)
        flight,service=float(row['Flight Time']),float(row['Service Time'])
        previous[h]=float(row['Completion Time'])
        if flight>0:
            ax.broken_barh([(start,flight)],(h-.32,.64),facecolors=colors['travel'])
        task=row['Status']=='Task Executed'
        ax.broken_barh([(start+flight,service)],(h-.32,.64),
                      facecolors=colors['service' if task else 'recovery'])
        if task:
            ax.text(start+flight+service/2,h,f"T{int(row['Task ID'])}",ha='center',va='center',
                    fontsize=5.3 if small else 6.,color='white')
    if not METHOD_RESULTS[method][1]['has_incumbent']:
        ax.text(.5,.5,'No solver incumbent',transform=ax.transAxes,ha='center',fontsize=8)
    helos=sorted(helo_to_int(h) for h in helicopters)
    ax.set_yticks(helos,[f'H{h}' for h in helos])
    ax.set_ylim(min(helos)-.6,max(helos)+.6)
    ax.set_xlim(common_mission_xlim())
    ax.set_title(method,fontsize=8.5,loc='left')
    ax.grid(axis='x',linestyle='--',linewidth=.4,alpha=.5)
    return [mpatches.Patch(color=colors['travel'],label='Flight'),
            mpatches.Patch(color=colors['service'],label='Task service'),
            mpatches.Patch(color=colors['recovery'],label='Return recovery')]


schedule_names=['gantt_greedy_ttap','gantt_greedy_density_ttap','gantt_exact_ttap']
for (method,(frame,_)),filename in zip(METHOD_RESULTS.items(),schedule_names):
    fig,ax=plt.subplots(figsize=(7.2,2.6))
    fig.subplots_adjust(left=.07,right=.99,bottom=.20,top=.76)
    handles=draw_schedule(ax,frame,method)
    ax.set_xlabel('Mission time (min)')
    fig.legend(handles=handles,loc='upper center',ncol=3,frameon=False,fontsize=7)
    save_publication_figure(fig,filename)

fig,axes=plt.subplots(3,1,figsize=(7.2,4.1),sharex=True)
fig.subplots_adjust(left=.07,right=.99,bottom=.13,top=.86,hspace=.48)
for ax,(method,(frame,_)) in zip(axes,METHOD_RESULTS.items()):
    handles=draw_schedule(ax,frame,method,small=True)
axes[-1].set_xlabel('Mission time (min)')
fig.legend(handles=handles,loc='upper center',ncol=3,frameon=False,fontsize=7)
save_publication_figure(fig,'gantt_comparison_three_methods')


### 8. Beneficio acumulado:

In [ ]:
def cumulative_points(frame,end):
    rows=frame[frame['Status'].eq('Task Executed')]
    grouped=rows.groupby('Completion Time')['Benefit'].sum().sort_index()
    x=np.r_[0.,grouped.index.to_numpy(dtype=float),end]
    y=np.r_[0.,grouped.cumsum().to_numpy(dtype=float),float(grouped.sum())]
    return x,y


end=common_mission_xlim()[1]
ymax=min(1.05,max(BENEFIT_Y_MAX,max((s['benefit'] for _,s in METHOD_RESULTS.values()
                                  if s['has_incumbent']),default=0.)+.05))
line_styles=['--','-.','-']
for index,(method,(frame,summary)) in enumerate(METHOD_RESULTS.items()):
    fig,ax=plt.subplots(figsize=(3.5,2.4),layout='constrained')
    if summary['has_incumbent']:
        x,y=cumulative_points(frame,end)
        ax.step(x,y,where='post',linewidth=1.1,color=METHOD_COLORS.get(method,'#1f77b4'))
    else:
        ax.text(.5,.5,'No solver incumbent',transform=ax.transAxes,ha='center')
    ax.set(xlim=(0,end),ylim=(0,ymax),xlabel='Mission time (min)',ylabel='Cumulative normalized benefit')
    ax.set_title(method,fontsize=8.5)
    ax.grid(axis='y',linestyle='--',linewidth=.45,alpha=.5)
    filename=['cumulative_benefit_greedy_ttap','cumulative_benefit_greedy_density_ttap',
              'cumulative_benefit_exact_ttap'][index]
    save_publication_figure(fig,filename)

fig,ax=plt.subplots(figsize=(7.2,3.),layout='constrained')
for style,(method,(frame,summary)) in zip(line_styles,METHOD_RESULTS.items()):
    if summary['has_incumbent']:
        x,y=cumulative_points(frame,end)
        ax.step(x,y,where='post',linestyle=style,linewidth=1.25,
                color=METHOD_COLORS.get(method,'#1f77b4'),label=method)
ax.set(xlim=(0,end),ylim=(0,ymax),xlabel='Mission time (min)',ylabel='Cumulative normalized benefit')
ax.grid(axis='y',linestyle='--',linewidth=.45,alpha=.5)
ax.legend(frameon=False,fontsize=7)
gbx,gby=cumulative_points(df_greedy_benefit,end)
gdx,gdy=cumulative_points(df_greedy_density,end)
if np.array_equal(gbx,gdx) and np.allclose(gby,gdy,atol=1e-12,rtol=0):
    ax.text(.98,.05,'Greedy Benefit and Density coincide',transform=ax.transAxes,
            ha='right',va='bottom',fontsize=7,color='#555555')
save_publication_figure(fig,'cumulative_benefit_comparison_three_methods')
# Vector alias preserves an existing Overleaf filename while adding Density.
import shutil
shutil.copyfile(Path(FIGURES_DIR)/'cumulative_benefit_comparison_three_methods.pdf',
                Path(FIGURES_DIR)/'cumulative_benefit_comparison_greedy_vs_exact.pdf')


### 9. Métricas comparativas:

In [ ]:
def gini_coefficient(values):
    values = np.array(values, dtype=float)

    if len(values) == 0:
        return np.nan

    if np.all(values == 0):
        return 0.0

    values = np.sort(values)
    n = len(values)
    cumulative = np.cumsum(values)

    return (n + 1 - 2 * np.sum(cumulative) / cumulative[-1]) / n


def compute_ttap_metrics(df, method_name, runtime_sec=None, optimal=False, mip_gap=None, exact_benefit_reference=None):
    df = df.copy()

    df["Helo ID"] = df["Helo ID"].apply(helo_to_int)

    task_df = df[df["Status"] == "Task Executed"].copy()
    idle_df = df[df["Status"] != "Task Executed"].copy()

    total_tasks = len(tasks)
    
    accumulated_benefit = task_df["Benefit"].sum() if not task_df.empty else 0.0
    
    # Tareas físicamente ejecutadas,
    # aunque hayan llegado tarde y tengan beneficio cero.
    physical_completed_tasks = task_df["Task ID"].nunique()
    
    # Tareas con beneficio estrictamente positivo.
    positive_task_df = task_df[task_df["Temporal Satisfaction"] > 0.0].copy()
    positive_completed_tasks = positive_task_df["Task ID"].nunique()
    
    # Tareas operacionalmente relevantes.
    # Esto elimina tareas con beneficio positivo, pero casi nulo.
    relevant_task_df = task_df[task_df["Temporal Satisfaction"]>= EFFECTIVE_SATISFACTION_THRESHOLD].copy()
    relevant_completed_tasks = relevant_task_df["Task ID"].nunique()
    
    physical_completion_rate = physical_completed_tasks / total_tasks
    positive_completion_rate = positive_completed_tasks / total_tasks
    relevant_completion_rate = relevant_completed_tasks / total_tasks

    # En escenario estático, todas las tareas están disponibles desde t=0.
    # Por tanto, response time = completion time.
    avg_response_time = task_df["Completion Time"].mean() if not task_df.empty else np.nan
    max_response_time = task_df["Completion Time"].max() if not task_df.empty else np.nan

    avg_activity_duration = task_df["Total Task Time"].mean() if not task_df.empty else np.nan
    max_activity_duration = task_df["Total Task Time"].max() if not task_df.empty else np.nan

    makespan = df["Completion Time"].max() if not df.empty else 0.0

    total_task_flight_time = task_df["Flight Time"].sum() if not task_df.empty else 0.0
    total_return_flight_time = idle_df["Flight Time"].sum() if not idle_df.empty else 0.0
    total_flight_time = total_task_flight_time + total_return_flight_time

    total_task_service_time = task_df["Service Time"].sum() if not task_df.empty else 0.0
    total_recovery_time = idle_df["Service Time"].sum() if not idle_df.empty else 0.0

    # Utilización de recursos:
    # - productive utilization: vuelo a tarea + servicio de tarea
    # - active utilization: vuelo total + servicio de tarea
    #   recuperación se considera no productiva.
    total_available_time = makespan * len(helicopters) if makespan > 0 else np.nan

    productive_time = total_task_flight_time + total_task_service_time
    active_time = total_flight_time + total_task_service_time

    productive_utilization = productive_time / total_available_time if total_available_time else np.nan
    active_utilization = active_time / total_available_time if total_available_time else np.nan

    # Operational cost proxy:
    operational_cost_proxy = total_flight_time + total_recovery_time

    # Load balancing
    helo_ids = sorted(
        helo_to_int(h)
        for h in helicopters.keys()
    )

    tasks_by_helo = (
        task_df.groupby("Helo ID")["Task ID"]
        .count()
        .reindex(helo_ids, fill_value=0)
    )

    workload_by_helo = (
        df.groupby("Helo ID")["Total Task Time"]
        .sum()
        .reindex(helo_ids, fill_value=0.0)
    )

    task_count_std = tasks_by_helo.std()
    task_count_cv = tasks_by_helo.std() / tasks_by_helo.mean() if tasks_by_helo.mean() > 0 else np.nan
    task_count_gini = gini_coefficient(tasks_by_helo.values)

    workload_std = workload_by_helo.std()
    workload_cv = workload_by_helo.std() / workload_by_helo.mean() if workload_by_helo.mean() > 0 else np.nan
    workload_gini = gini_coefficient(workload_by_helo.values)

    if exact_benefit_reference is not None and exact_benefit_reference > 0:
        optimality_gap_vs_exact = (exact_benefit_reference - accumulated_benefit) / exact_benefit_reference
    else:
        optimality_gap_vs_exact = np.nan

    metrics = {
        "Method": method_name,

        # Solution quality
        "Accumulated Benefit": accumulated_benefit,
        "Optimality Gap vs Exact": optimality_gap_vs_exact,

        # Task completion
        "Physical Completed Tasks": physical_completed_tasks,
        "Physical Completion Rate": physical_completion_rate,
        "Positive Benefit Tasks": positive_completed_tasks,
        "Positive Benefit Completion Rate": positive_completion_rate,
        "Operationally Relevant Tasks": relevant_completed_tasks,
        "Operationally Relevant Completion Rate": relevant_completion_rate,

        # Response time
        "Average Response Time": avg_response_time,
        "Maximum Response Time": max_response_time,
        "Average Activity Duration": avg_activity_duration,
        "Maximum Activity Duration": max_activity_duration,

        # Resource utilization
        "Makespan": makespan,
        "Total Available Fleet Time": total_available_time,
        "Productive Time": productive_time,
        "Active Time": active_time,
        "Productive Utilization": productive_utilization,
        "Active Utilization": active_utilization,

        # Operational cost proxies
        "Total Task Flight Time": total_task_flight_time,
        "Total Return Flight Time": total_return_flight_time,
        "Total Flight Time": total_flight_time,
        "Total Recovery Time": total_recovery_time,
        "Operational Cost Proxy": operational_cost_proxy,

        # Load balancing
        "Task Count Std": task_count_std,
        "Task Count CV": task_count_cv,
        "Task Count Gini": task_count_gini,
        "Workload Std": workload_std,
        "Workload CV": workload_cv,
        "Workload Gini": workload_gini,

        # Computational efficiency
        "Runtime sec": runtime_sec,
        "Optimal Certified": optimal,
        "MIP Gap": mip_gap,
    }

    detail = {
        "tasks_by_helo": tasks_by_helo,
        "workload_by_helo": workload_by_helo,
    }

    return metrics, detail


metric_rows=[]
metric_details={}
for method,(frame,summary) in METHOD_RESULTS.items():
    metrics,details=compute_ttap_metrics(frame,method_name=method,runtime_sec=summary['runtime_sec'],
                                        optimal=summary['optimal'],mip_gap=summary['mip_gap'],
                                        exact_benefit_reference=certified_reference)
    metrics['Objective Mode']=OBJECTIVE_MODE
    metrics['Has Incumbent']=summary['has_incumbent']
    if summary['has_incumbent']:
        executed_ids=frame.loc[frame['Status'].eq('Task Executed'),'Task ID'].astype(int).tolist()
        for cls,field,name in [('medical','med','Medical Evacuees'),
                               ('personnel','pers','Personnel Transported'),
                               ('cargo','load','Cargo Delivered kg')]:
            selected=[tasks[i] for i in executed_ids
                      if OBJECTIVE_MODE=='multidimensional' or tasks[i]['priority_class']==cls]
            metrics[name]=sum(t[field] for t in selected)
    else:
        for key in metrics:
            if key not in {'Method','Objective Mode','Has Incumbent','Runtime sec','Optimal Certified','MIP Gap'}:
                metrics[key]=np.nan
    metric_rows.append(metrics)
    metric_details[method]=details
df_metrics=pd.DataFrame(metric_rows)
df_metrics.to_csv(Path(FIGURES_DIR)/'talcahuano_metrics.csv',index=False)
display(df_metrics.T)


### 10. Tabla de balance por Helo:

In [ ]:
def build_helo_balance_table(df, method_name):
    df = df.copy()
    df["Helo ID"] = df["Helo ID"].apply(helo_to_int)

    task_df = df[df["Status"] == "Task Executed"].copy()

    balance = []

    for h in sorted(helo_to_int(x) for x in helicopters):
        df_h = df[df["Helo ID"] == h]
        task_h = task_df[task_df["Helo ID"] == h]

        balance.append({
            "Method": method_name,
            "Helo ID": f"H{h}",
            "Tasks Executed": task_h["Task ID"].nunique(),
            "Benefit": task_h["Benefit"].sum(),
            "Flight Time": df_h["Flight Time"].sum(),
            "Task Service Time": task_h["Service Time"].sum(),
            "Recovery Time": df_h[df_h["Status"] != "Task Executed"]["Service Time"].sum(),
            "Workload Time": df_h["Total Task Time"].sum(),
            "Last Completion Time": df_h["Completion Time"].max() if not df_h.empty else 0.0,
        })

    return pd.DataFrame(balance)


balance_frames=[]
for method,(frame,summary) in METHOD_RESULTS.items():
    table=build_helo_balance_table(frame,method)
    if not summary['has_incumbent']:
        table.loc[:,table.columns.difference(['Method','Helo ID'])]=np.nan
    balance_frames.append(table)
df_balance=pd.concat(balance_frames,ignore_index=True)
df_balance.to_csv(Path(FIGURES_DIR)/'talcahuano_helo_balance.csv',index=False)
display(df_balance)


_____